# Day 06 - 1교시: 네트워크 기초
> Kafka와 Docker를 위한 사전 지식

## 🎯 학습 목표
이 파트를 마치면 다음 질문에 답할 수 있습니다:

- "포트 5000번으로 접속한다"는 게 무슨 뜻이야?
- `localhost`랑 `0.0.0.0`은 뭐가 달라?
- Docker에서 `-p 5001:5001`은 왜 필요해?
- `docker-compose`에서 서비스 이름으로 통신이 되는 이유는?

---
## 📚 전체 학습 흐름

```
네트워크 기초 (이 교시)
    │
    ├─ 프로그램/프로세스/포트 이해
    │       ↓
    │   [Flask 실습에서] app.run(port=5001) 의미 파악
    │
    ├─ 0.0.0.0 vs 127.0.0.1 이해
    │       ↓
    │   [Docker 실습에서] host='0.0.0.0' 왜 필요한지 파악
    │
    ├─ 포트 포워딩/매핑 이해
    │       ↓
    │   [docker-compose] ports: "5001:5001" 의미 파악
    │
    └─ 브리지 네트워크 & 서비스 이름 이해
            ↓
        [Kafka 설정에서] KAFKA_LISTENERS, KAFKA_ADVERTISED_LISTENERS 이해
```

---
## 📚 1. 프로그램, 프로세스, 포트

### 1.1 프로그램 vs 프로세스

![프로그램 vs 프로세스](https://velog.velcdn.com/images/moonblue/post/e40a03c4-8f30-4827-b812-1daa1d9d2f19/image.jpeg)

```
┌─────────────────────────────────────────────────────────────┐
│                         컴퓨터                              │
│                                                             │
│   📁 프로그램 (파일)          🔄 프로세스 (실행 중)           │
│   ─────────────────          ─────────────────────          │
│   chrome.exe                 Chrome (PID: 1234)            │
│   python.exe                 python shop.py (PID: 5678)    │
│   code.exe                   VS Code (PID: 9012)           │
│                                                             │
│   "설치만 된 상태"            "실제로 돌아가는 상태"          │
└─────────────────────────────────────────────────────────────┘
```

| 구분 | 프로그램 | 프로세스 |
|------|----------|----------|
| 상태 | 디스크에 저장된 파일 | 메모리에서 실행 중 |
| 비유 | 요리 레시피 | 실제로 요리하는 중 |
| 개수 | 하나의 프로그램 | 여러 프로세스 가능 |

> **예시:** `python`이라는 프로그램 하나로 `shop.py`, `customer.py`를 동시에 실행하면 2개의 프로세스가 생성

### 1.2 포트(Port)란?

컴퓨터 한 대에서 여러 프로세스가 동시에 네트워크 통신을 합니다.
**포트**: "어떤 프로세스에게 전달할지" 구분하는 번호

```
┌─────────────────────────────────────────────────────────────┐
│                    내 컴퓨터 (IP: 192.168.0.10)              │
│                                                             │
│     ┌─────────────┐  ┌─────────────┐  ┌─────────────┐      │
│     │   Chrome    │  │  Flask 앱   │  │   Kafka     │      │
│     │  (브라우저)  │  │  (shop.py)  │  │  (브로커)    │      │
│     └──────┬──────┘  └──────┬──────┘  └──────┬──────┘      │
│            │                │                │              │
│         :443             :5001            :9092            │
│            │                │                │              │
│     ───────┴────────────────┴────────────────┴───────      │
│                      네트워크 카드                          │
└─────────────────────────────────────────────────────────────┘

외부에서 192.168.0.10:5001로 요청 → Flask 앱이 받음
외부에서 192.168.0.10:9092로 요청 → Kafka가 받음
```

| 비유 | 실제 |
|------|------|
| 아파트 주소 | IP 주소 (192.168.0.10) |
| 호수 (101호, 102호) | 포트 번호 (5001, 9092) |
| 택배 기사 | 네트워크 패킷 |

### 1.3 포트 번호의 범위

![각 포트번호에 대한 설명](https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FbtuFOs%2FbtsBXKOeLWR%2FAAAAAAAAAAAAAAAAAAAAAKSyDrznZT5eLRp7fWu90DLMHv46qOXSu339I3VbmP7g%2Fimg.png%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1769871599%26allow_ip%3D%26allow_referer%3D%26signature%3DenOrCTDzYFf4TbtYDDnAeuYyrfI%253D)

```
┌───────────────────────────────────────────────────────────────┐
│                    포트 번호: 0 ~ 65535                        │
├───────────────────────────────────────────────────────────────┤
│  0 ~ 1023      │  Well-Known Ports (예약됨)                   │
│                │  HTTP(80), HTTPS(443), SSH(22) 등            │
│                │  ⚠️ 관리자 권한 필요                          │
├────────────────┼──────────────────────────────────────────────┤
│  1024 ~ 49151  │  Registered Ports (등록됨)                   │
│                │  MySQL(3306), PostgreSQL(5432) 등            │
│                │  ✅ 일반 사용자 사용 가능                      │
├────────────────┼──────────────────────────────────────────────┤
│  49152 ~ 65535 │  Dynamic/Private Ports                       │
│                │  OS가 임시로 할당하는 용도                     │
└───────────────────────────────────────────────────────────────┘
```

### 1.4 자주 사용하는 포트 번호

![자주 사용하는 포트 번호](https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FJUi2b%2FbtsB3cWwnq9%2FAAAAAAAAAAAAAAAAAAAAAM8csDmQj629G7eBzg44iev1JSmNK8aqGExiR9_-B9GS%2Fimg.png%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1769871599%26allow_ip%3D%26allow_referer%3D%26signature%3DoIFzYit%252Fgbey79AV0D%252Fw2O9heyk%253D)

| 포트 | 서비스 | 우리 실습에서 |
|------|--------|---------------|
| 80 | HTTP (웹) | - |
| 443 | HTTPS (보안 웹) | - |
| 5000 | Flask 기본 | 주문 서비스 |
| 5001 | - | 재고 서비스 |
| 8080 | 웹 앱 대체 포트 | Kafka UI |
| 9092 | Kafka 기본 | Kafka 브로커 |
| 9093 | - | Kafka 컨트롤러 |

### 1.5 뒤에서 이렇게 연결됩니다

```python
# Flask 실습에서 보게 될 코드
app.run(port=5001)  # ← "5001번 포트에서 대기해!"
```

```yaml
# Kafka 설정에서 보게 될 내용
KAFKA_LISTENERS: PLAINTEXT://localhost:9092  # ← "9092번 포트에서 대기해!"
```

```yaml
# docker-compose.yml에서 보게 될 내용
services:
  order:
    ports:
      - "5000:5000"   # ← 주문 서비스는 5000번

  inventory:
    ports:
      - "5001:5001"   # ← 재고 서비스는 5001번
```

### 💻 실습: 프로세스와 포트 확인하기

터미널에서 다음 명령어를 실행해 현재 시스템의 프로세스와 포트를 확인해봅시다.

#### 1. 현재 실행 중인 프로세스 확인

```bash
ps aux
```

**실행 결과 예시:**
```
USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root           1  0.0  0.0   2268  1128 ?        Ss   15:05   0:00 tail -f /dev/null
root        3298  0.2  0.2  23876 16832 ?        Ss   15:06   0:00 python3 -m http.server 5001 --bind 0.0.0.0
root        3304  0.1  0.2  23876 16788 ?        Ss   15:06   0:00 python3 -m http.server 5002 --bind 127.0.0.1
root        3322  0.0  0.0   6452  2536 ?        Rs   15:07   0:00 ps aux
```

> **해석**: PID(프로세스 ID)별로 어떤 명령어가 실행 중인지 확인할 수 있습니다.

#### 2. 열린 포트 확인 (ss 명령어)

```bash
ss -tuln
```

**실행 결과 예시:**
```
Netid State  Recv-Q Send-Q Local Address:Port Peer Address:PortProcess
tcp   LISTEN 0      5            0.0.0.0:5001      0.0.0.0:*
tcp   LISTEN 0      5          127.0.0.1:5002      0.0.0.0:*
```

> **해석**:
> - `0.0.0.0:5001` - 모든 IP에서 5001번 포트로 접근 가능
> - `127.0.0.1:5002` - 로컬에서만 5002번 포트로 접근 가능

#### 3. 열린 포트 확인 (netstat 명령어 - 대안)

```bash
netstat -tuln
```

**실행 결과 예시:**
```
Active Internet connections (only servers)
Proto Recv-Q Send-Q Local Address           Foreign Address         State
tcp        0      0 0.0.0.0:5001            0.0.0.0:*               LISTEN
tcp        0      0 127.0.0.1:5002          0.0.0.0:*               LISTEN
```

> **옵션 설명**:
> - `-t`: TCP 연결만 표시
> - `-u`: UDP 연결만 표시
> - `-l`: LISTEN 상태(대기 중)만 표시
> - `-n`: 숫자로 표시 (DNS 변환 없이)

---
## 📚 2. 0.0.0.0 vs 127.0.0.1

### 2.1 두 주소의 차이

이 개념은 Docker 실습에서 **반드시** 이해해야 합니다.

```
┌─────────────────────────────────────────────────────────────┐
│                         내 컴퓨터                            │
│                                                             │
│   ┌─────────────────────────────────────────────────────┐   │
│   │              127.0.0.1 (localhost)                  │   │
│   │              "나 자신만을 위한 주소"                  │   │
│   │                                                     │   │
│   │    🏠 내부 전용 - 외부에서 절대 접근 불가             │   │
│   └─────────────────────────────────────────────────────┘   │
│                                                             │
│   ┌─────────────────────────────────────────────────────┐   │
│   │              0.0.0.0 ("모든 인터페이스")              │   │
│   │              "누구든 오세요"                          │   │
│   │                                                     │   │
│   │    🌐 내부 + 외부 모두 접근 가능                      │   │
│   └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 2.2 비유로 이해하기

```
127.0.0.1 (localhost)
├─ 집 안에서만 통하는 인터폰
├─ 집 밖 사람은 연결 불가
└─ "나 혼자 테스트할 때"

0.0.0.0
├─ 집 현관문 + 뒷문 + 모든 창문 열어둠
├─ 어디서든 들어올 수 있음
└─ "외부에서도 접근 허용할 때"
```

### 2.3 실제 코드에서의 차이

```python
# ❌ 로컬에서만 접근 가능
app.run(host='127.0.0.1', port=5001)
# 같은 컴퓨터의 브라우저에서만 http://localhost:5001 접속 가능

# ✅ 외부(다른 컴퓨터, Docker 컨테이너)에서도 접근 가능
app.run(host='0.0.0.0', port=5001)
# 네트워크 내 다른 컴퓨터에서도 http://192.168.0.10:5001 접속 가능
```

### 2.4 뒤에서 이렇게 연결됩니다

```python
# Docker 실습에서 보게 될 코드 (shop/app.py)
if __name__ == '__main__':
    # host='0.0.0.0' → Docker 컨테이너 밖에서도 접근 가능하게!
    app.run(host='0.0.0.0', port=5001)
```

> **왜 Docker에서는 0.0.0.0이 필수인가?**
> Docker 컨테이너는 "별도의 컴퓨터"처럼 동작합니다.
> `127.0.0.1`로 설정하면 컨테이너 안에서만 접근 가능하고,
> 호스트 PC나 다른 컨테이너에서는 접근할 수 없습니다.

### 💻 실습: 0.0.0.0 vs 127.0.0.1 차이 확인하기

#### 1. 네트워크 인터페이스 확인

```bash
ip addr
```

**실행 결과 예시:**
```
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP
    link/ether 00:15:5d:xx:xx:xx brd ff:ff:ff:ff:ff:ff
    inet 172.20.10.5/20 brd 172.20.15.255 scope global eth0
       valid_lft forever preferred_lft forever
```

> **해석**:
> - `lo` (loopback): 127.0.0.1 - 자기 자신만 접근 가능
> - `eth0`: 172.20.10.5 - 외부에서도 접근 가능한 IP

#### 2. 바인딩 주소별 접근 차이 확인

먼저 두 개의 서버를 다른 주소로 실행합니다:

```bash
# 터미널 1: 0.0.0.0으로 바인딩 (모든 곳에서 접근 가능)
python3 -m http.server 5001 --bind 0.0.0.0

# 터미널 2: 127.0.0.1로 바인딩 (로컬만 접근 가능)
python3 -m http.server 5002 --bind 127.0.0.1
```

#### 3. 열린 포트와 바인딩 주소 확인

```bash
ss -tulnp | grep python
```

**실행 결과 예시:**
```
tcp  LISTEN 0  5  0.0.0.0:5001  0.0.0.0:*  users:(("python3",pid=3298,fd=3))
tcp  LISTEN 0  5  127.0.0.1:5002  0.0.0.0:*  users:(("python3",pid=3304,fd=3))
```

> **핵심 차이점**:
> - `0.0.0.0:5001` → 외부 네트워크에서도 접근 가능
> - `127.0.0.1:5002` → 같은 컴퓨터에서만 접근 가능

---
## 📚 3. 소켓(Socket): IP와 Port의 결합

### 3.1 소켓이란?

**소켓 = IP 주소 + 포트 번호**

네트워크 통신의 "끝점(endpoint)"을 정의합니다.

```
┌─────────────────────────────────────────────────────────────┐
│                      소켓 (Socket)                          │
│                                                             │
│         192.168.0.10  :  5001                               │
│         ─────────────    ────                               │
│            IP 주소       포트                                │
│          (어떤 컴퓨터)   (어떤 프로세스)                       │
│                                                             │
│         = "192.168.0.10 컴퓨터의 5001번 프로세스"            │
└─────────────────────────────────────────────────────────────┘
```

### 3.2 통신은 소켓과 소켓 사이에서 일어남

```
클라이언트 (손님)                              서버 (가게)
┌──────────────────┐                    ┌──────────────────┐
│ 192.168.0.20     │                    │ 192.168.0.10     │
│      :54321      │ ───── 요청 ─────▶ │      :5001       │
│  (임시 포트)      │                    │  (Flask 앱)      │
│                  │ ◀──── 응답 ────── │                  │
└──────────────────┘                    └──────────────────┘

통신 경로: 192.168.0.20:54321 ↔ 192.168.0.10:5001
```

### 💻 실습: 소켓 연결 확인하기

#### 1. 현재 소켓 연결 상태 확인

```bash
ss -tp
```

**실행 결과 예시 (연결이 있을 때):**
```
State  Recv-Q Send-Q Local Address:Port   Peer Address:Port Process
ESTAB  0      0      172.20.10.5:5001     172.20.10.1:54321
```

> **해석**: 172.20.10.5:5001(서버)과 172.20.10.1:54321(클라이언트) 간 연결이 수립됨

#### 2. curl로 연결 테스트

```bash
# 5001번 포트로 접속 테스트
curl localhost:5001
```

**실행 결과 예시:**
```html
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd">
<html>
<head>
<meta http-equiv="Content-Type" content="text/html; charset=utf-8">
<title>Directory listing for /</title>
```

> **해석**: HTTP 서버가 응답하면 연결 성공!

#### 3. ping으로 네트워크 연결 확인

```bash
ping -c 3 localhost
```

**실행 결과 예시:**
```
PING localhost(localhost (::1)) 56 data bytes
64 bytes from localhost (::1): icmp_seq=1 ttl=64 time=0.730 ms
64 bytes from localhost (::1): icmp_seq=2 ttl=64 time=0.157 ms
64 bytes from localhost (::1): icmp_seq=3 ttl=64 time=0.076 ms

--- localhost ping statistics ---
3 packets transmitted, 3 received, 0% packet loss, time 2069ms
```

> **해석**: 3개 패킷 모두 응답 받음 → 네트워크 연결 정상

---
## 📚 4. 포트 포워딩과 포트 매핑

### 4.1 포트 포워딩 개념

**포트 포워딩 = "들어온 요청을 다른 곳으로 전달"**

```
┌─────────────────────────────────────────────────────────────┐
│                       공유기 (Router)                        │
│                                                             │
│   외부 인터넷                              내부 네트워크       │
│   ───────────                              ──────────────    │
│                                                             │
│   공인 IP:80  ──────────────────────▶  192.168.0.10:5000   │
│   (누구나 접근)     포트 포워딩           (내 Flask 앱)       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 4.2 Docker에서의 포트 매핑

Docker 컨테이너는 **격리된 네트워크**를 가집니다.
외부(호스트 PC)에서 접근하려면 **포트 매핑**이 필요합니다.

```
┌──────────────────────────────────────────────────────────────┐
│                         호스트 PC                            │
│                                                              │
│    브라우저에서                                               │
│    localhost:5001 접속                                        │
│          │                                                   │
│          │ 포트 매핑                                          │
│          │ (5001 → 5001)                                     │
│          ▼                                                   │
│    ┌─────────────────────────────────────────────────────┐   │
│    │              Docker 컨테이너                         │   │
│    │                                                     │   │
│    │         Flask 앱이 0.0.0.0:5001에서 대기            │   │
│    │                                                     │   │
│    └─────────────────────────────────────────────────────┘   │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

### 4.3 포트 매핑 문법

```yaml
ports:
  - "호스트포트:컨테이너포트"
  - "5001:5001"      # 같은 번호 사용 (가장 일반적)
  - "8080:5001"      # 다른 번호도 가능
  - "5001"           # 호스트 포트 생략 시 랜덤 할당
```

### 4.4 왜 두 개의 포트가 필요한가?

```
시나리오: 같은 앱을 2개 실행하고 싶다면?

┌─────────────────────────────────────────────────────────────┐
│                       호스트 PC                              │
│                                                             │
│   localhost:5001 ──▶ 컨테이너A의 5001 (Flask 앱 인스턴스1)   │
│   localhost:5002 ──▶ 컨테이너B의 5001 (Flask 앱 인스턴스2)   │
│                                                             │
│   컨테이너 안에서는 둘 다 5001번 사용                         │
│   호스트에서 구분하기 위해 다른 포트로 매핑                    │
└─────────────────────────────────────────────────────────────┘
```

---
## 📚 5. 브리지 네트워크와 서비스 이름

### 5.1 Docker 네트워크의 필요성

Docker 컨테이너는 각각 **격리된 환경**입니다.
서로 통신하려면 **네트워크**로 연결해야 합니다.

```
┌─────────────────────────────────────────────────────────────┐
│                    Docker 브리지 네트워크                    │
│                    (가상의 스위치/공유기)                     │
│                                                             │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐             │
│   │ 컨테이너A │    │ 컨테이너B │    │ 컨테이너C │             │
│   │  :5000   │    │  :5001   │    │  :5002   │             │
│   │ 172.17.  │    │ 172.17.  │    │ 172.17.  │             │
│   │  0.2     │    │  0.3     │    │  0.4     │             │
│   └────┬─────┘    └────┬─────┘    └────┬─────┘             │
│        │               │               │                   │
│   ─────┴───────────────┴───────────────┴─────              │
│              브리지 네트워크 (172.17.0.0/16)                 │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 5.2 브리지 네트워크 특징

| 특징 | 설명 |
|------|------|
| 자동 생성 | `docker-compose up` 하면 자동으로 네트워크 생성 |
| IP 자동 할당 | 각 컨테이너에 172.17.x.x 형태의 IP 부여 |
| 컨테이너 간 통신 | 같은 네트워크 안에서는 서로 접근 가능 |
| 호스트와 격리 | 호스트 PC와는 기본적으로 분리됨 |

### 5.3 서비스 이름 = DNS 이름

Docker Compose의 **가장 강력한 기능** 중 하나입니다.
서비스 이름이 곧 **호스트 이름(DNS)**이 됩니다.

```yaml
services:
  order:        # ← 이 이름이 곧 주소가 됨!
    build: ./order

  inventory:    # ← "inventory"로 접근 가능
    build: ./inventory

  shipping:     # ← "shipping"으로 접근 가능
    build: ./shipping
```

### 5.4 IP 주소 대신 서비스 이름 사용

```
┌─────────────────────────────────────────────────────────────┐
│              Docker Compose 네트워크                         │
│                                                             │
│   ❌ 옛날 방식 (IP 직접 사용)                                 │
│   requests.get("http://172.17.0.3:5001/check")              │
│   → IP가 바뀌면 코드 수정 필요                                │
│                                                             │
│   ✅ 서비스 이름 사용                                        │
│   requests.get("http://inventory:5001/check")               │
│   → IP가 바뀌어도 자동으로 해결됨!                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 5.5 내부 DNS 동작 원리

```
order 컨테이너에서 "inventory:5001" 요청
        │
        ▼
Docker 내부 DNS가 "inventory"를 IP로 변환
        │
        ▼
172.17.0.3:5001로 요청 전달
        │
        ▼
inventory 컨테이너가 응답
```

### 5.6 뒤에서 이렇게 연결됩니다

```python
# order/app.py에서 다른 서비스 호출
# IP 주소가 아닌 서비스 이름을 사용!

requests.get("http://inventory:5001/check")   # ← 서비스 이름!
requests.get("http://shipping:5002/schedule") # ← 서비스 이름!
requests.get("http://notification:5003/send") # ← 서비스 이름!
```

```yaml
# Kafka 설정에서도 마찬가지
KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka-controller:9093
#                                  ^^^^^^^^^^^^^^^^
#                                  컨테이너 이름/서비스 이름
```

### 💻 실습: Docker 네트워크 확인하기

> **참고**: 이 실습은 Docker가 설치된 환경에서 진행합니다.

#### 1. Docker 네트워크 목록 확인

```bash
docker network ls
```

**실행 결과 예시:**
```
NETWORK ID     NAME                  DRIVER    SCOPE
a1b2c3d4e5f6   bridge                bridge    local
g7h8i9j0k1l2   host                  host      local
m3n4o5p6q7r8   myapp_default         bridge    local
```

> **해석**: `myapp_default`는 docker-compose로 생성된 브리지 네트워크

#### 2. 특정 네트워크의 상세 정보 확인

```bash
docker network inspect bridge
```

**실행 결과 예시 (주요 부분):**
```json
[
    {
        "Name": "bridge",
        "Driver": "bridge",
        "IPAM": {
            "Config": [
                {
                    "Subnet": "172.17.0.0/16",
                    "Gateway": "172.17.0.1"
                }
            ]
        },
        "Containers": {
            "abc123...": {
                "Name": "my-container",
                "IPv4Address": "172.17.0.2/16"
            }
        }
    }
]
```

> **해석**: 브리지 네트워크는 172.17.0.0/16 대역을 사용하고, 컨테이너에 IP가 할당됨

#### 3. 컨테이너 내부에서 DNS 확인 (docker-compose 환경)

```bash
# 컨테이너 내부에서 실행
cat /etc/hosts
```

**실행 결과 예시:**
```
127.0.0.1       localhost
172.17.0.2      abc123def456    my-container
```

---
## 📚 6. 종합 정리: 이 개념들이 어디서 쓰이나

### 6.1 Flask 실습 (2교시)

```python
app.run(port=5001)  # 포트 개념
```
- **포트**: 5001번에서 대기
- **소켓**: localhost:5001

### 6.2 Docker 실습 (2교시)

```python
app.run(host='0.0.0.0', port=5001)  # 0.0.0.0 개념
```

```dockerfile
# Dockerfile
EXPOSE 5001  # 컨테이너가 5001 사용한다고 명시 (문서화 목적)
```

```yaml
# docker-compose.yml
ports:
  - "5001:5001"  # 포트 매핑 개념
```

### 6.3 전체 시스템 실습 (2교시)

```yaml
services:
  order:
    ports:
      - "5000:5000"   # 외부 접근용 포트 매핑

  inventory:          # 포트 매핑 없음 - 내부 전용
  shipping:           # 포트 매핑 없음 - 내부 전용
  notification:       # 포트 매핑 없음 - 내부 전용
```

```python
# 서비스 이름으로 통신
requests.get("http://inventory:5001/check")  # 브리지 네트워크 + DNS
```

### 6.4 Kafka 실습 (4교시 이후)

```yaml
environment:
  # 리스너 설정 - 어디서 접속을 받을지
  KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
  #                          ───────────────
  #                          0.0.0.0 = 모든 곳에서 접속 허용

  # 광고 주소 - 클라이언트에게 알려줄 주소
  KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://localhost:9092
  #                                      ─────────────────
  #                                      "이 주소로 연결해!"

  # 컨트롤러 투표자 - 서비스 이름 사용
  KAFKA_CONTROLLER_QUORUM_VOTERS: 1@localhost:9093
  #                               ──────────────────
  #                               소켓 주소 형식
```

---
## ❓ FAQ

**Q1. `localhost`와 `127.0.0.1`은 같은 건가요?**

네, 거의 같습니다. `localhost`는 `127.0.0.1`의 별명(alias)입니다.
단, `localhost`는 DNS 조회가 필요하고, `127.0.0.1`은 직접 사용됩니다.

---

**Q2. 포트 충돌이 나면 어떻게 하나요?**

이미 다른 프로세스가 해당 포트를 사용 중입니다.
- 다른 포트 번호를 사용하거나
- 기존 프로세스를 종료하세요
- 확인 명령어: `lsof -i :5001` (Mac/Linux) 또는 `netstat -ano | findstr :5001` (Windows)

---

**Q3. Docker에서 포트 매핑을 안 하면 어떻게 되나요?**

호스트 PC에서 컨테이너에 접근할 수 없습니다.
하지만 같은 Docker 네트워크 안의 다른 컨테이너끼리는 통신 가능합니다.

---

**Q4. `0.0.0.0`으로 설정하면 보안에 문제가 없나요?**

개발 환경에서는 괜찮지만, 운영 환경에서는 방화벽 설정이 필요합니다.
외부 IP에서 접근 가능하므로 인증/인가 로직을 추가하세요.

※ 방화벽: "누가, 어디서, 어떤 포트로 접근하는지" 검사해서 허용/차단을 결정합니다.

---

**Q5. Docker Compose에서 서비스 이름으로 통신이 안 되면?**

같은 `docker-compose.yml` 파일에 정의된 서비스인지 확인하세요.
별도의 compose 파일이라면 같은 네트워크에 연결해야 합니다.

---
## 📝 퀴즈

### Q1. 포트(Port)의 역할은 무엇인가요?

- A) 컴퓨터의 IP 주소를 정하는 역할
- B) 한 컴퓨터에서 여러 프로세스를 구분하는 번호
- C) 네트워크 속도를 조절하는 역할
- D) 데이터를 암호화하는 역할

<details>
<summary>정답 보기</summary>

**정답: B**

포트는 하나의 IP 주소에서 여러 프로세스(프로그램)를 구분하기 위한 번호입니다.
아파트 주소(IP)에서 호수(포트)를 구분하는 것과 같습니다.
</details>

---

### Q2. Flask 앱을 Docker 컨테이너에서 실행할 때, 왜 `host='0.0.0.0'`으로 설정해야 하나요?

- A) 속도가 더 빨라지기 때문
- B) 보안이 강화되기 때문
- C) 컨테이너 외부에서도 접근 가능하게 하기 위해
- D) 포트 충돌을 방지하기 위해

<details>
<summary>정답 보기</summary>

**정답: C**

`127.0.0.1`은 로컬(자기 자신)에서만 접근 가능합니다.
Docker 컨테이너는 별도의 네트워크 환경이므로, 호스트 PC에서 접근하려면
`0.0.0.0`(모든 네트워크 인터페이스)으로 설정해야 합니다.
</details>

---

### Q3. `docker-compose.yml`에서 `ports: "8080:5000"`의 의미는?

- A) 컨테이너의 8080 포트를 호스트의 5000 포트로 연결
- B) 호스트의 8080 포트를 컨테이너의 5000 포트로 연결
- C) 8080과 5000 포트를 동시에 사용
- D) 포트 범위 8080~5000을 모두 열기

<details>
<summary>정답 보기</summary>

**정답: B**

형식: `"호스트포트:컨테이너포트"`
호스트의 8080번 포트로 접속하면 컨테이너의 5000번 포트로 전달됩니다.
브라우저에서 `localhost:8080` → 컨테이너 내부 Flask 앱(5000)
</details>

---

### Q4. Docker Compose에서 서비스 이름으로 통신이 가능한 이유는?

- A) 호스트 PC의 hosts 파일에 등록되기 때문
- B) Docker 내부 DNS가 서비스 이름을 IP로 변환해주기 때문
- C) 모든 컨테이너가 같은 IP를 공유하기 때문
- D) 인터넷 DNS 서버가 처리하기 때문

<details>
<summary>정답 보기</summary>

**정답: B**

Docker Compose는 같은 네트워크에 속한 서비스들에 대해 내부 DNS를 제공합니다.
서비스 이름(예: `inventory`)으로 요청하면 자동으로 해당 컨테이너의 IP로 변환됩니다.
</details>

---
## 📋 핵심 요약

| 개념 | 설명 | 사용 예시 |
|------|------|-----------|
| **포트** | 프로세스를 구분하는 번호 | `app.run(port=5001)` |
| **127.0.0.1** | 로컬 전용 주소 | 개발 중 테스트 |
| **0.0.0.0** | 모든 네트워크 인터페이스 | Docker 컨테이너 |
| **소켓** | IP + 포트 조합 | `localhost:5001` |
| **포트 매핑** | 호스트↔컨테이너 포트 연결 | `ports: "5001:5001"` |
| **브리지 네트워크** | 컨테이너 간 통신망 | docker-compose 자동 생성 |
| **서비스 이름** | 컨테이너 DNS 이름 | `http://inventory:5001` |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| "포트 5000에서 대기한다"는 무슨 뜻? | ☐ |
| Flask에서 `host='0.0.0.0'`은 왜 필요? | ☐ |
| `ports: "5001:5001"`의 의미는? | ☐ |
| `http://inventory:5001`이 작동하는 이유? | ☐ |
| Kafka의 `LISTENERS`와 `ADVERTISED_LISTENERS` 차이? | ☐ |

---


# Day 06 - 2교시: 동기식 통신의 한계 체험
> Kafka 도입 전, 기존 "전화 방식" 시스템의 문제점 체험

## 🎯 학습 목표
이 파트를 마치면 다음을 이해할 수 있습니다:

- Flask로 간단한 웹 서버를 만드는 방법
- 서비스 간 동기식 통신(HTTP 요청)의 동작 방식
- 동기식 통신의 문제점: 느림, 장애 전파, 데이터 유실
- **왜 Kafka 같은 메시지 큐가 필요한지** 체감

## 📚 전체 학습 흐름

```
Step 0: Flask가 뭐야? (웹 서버 기초)
    ↓
Step 1: 두 프로그램이 "전화"하기 (HTTP 통신)
    ↓
Step 2: Docker로 옮기기 (컨테이너화)
    ↓
Step 3: 전체 시스템 구성 & 문제 체험
    ↓
💡 깨달음: "중간에 우체국이 있으면 어떨까?" → Kafka!
```

---
## 🔑 핵심 용어: 동기식 vs 비동기식

### 동기식 통신 (Synchronous Communication)

> **정의**: 요청을 보내고 **응답이 올 때까지 대기**하는 통신 방식

```
동기식 = 전화 통화
─────────────────────────────────────────────────────
1. 전화를 건다 (요청)
2. 상대방이 받을 때까지 기다린다 (대기 - 블로킹)
3. 통화한다 (응답)
4. 전화를 끊고 다음 일을 한다

특징:
• 블로킹(Blocking): 응답 전까지 다른 작업 불가
• 즉시성: 바로 결과를 알 수 있음
• 의존성: 상대방이 있어야만 통신 가능
```

**실생활 예시:**
- 전화 통화: 상대방이 받아야 대화 가능
- 카페 주문: 음료 나올 때까지 카운터 앞에서 대기
- ATM 출금: 현금 나올 때까지 기다림

**프로그래밍 예시:**
- HTTP 요청/응답 (REST API)
- 데이터베이스 쿼리 실행
- 함수 호출 후 반환값 대기

### 비동기식 통신 (Asynchronous Communication)

> **정의**: 요청을 보내고 **응답을 기다리지 않고** 다음 작업을 진행하는 통신 방식

```
비동기식 = 우편/택배
─────────────────────────────────────────────────────
1. 편지를 우체통에 넣는다 (요청)
2. 바로 집에 간다 (대기 없이 다음 작업)
3. 나중에 답장이 온다 (응답은 언젠가)

특징:
• 논블로킹(Non-blocking): 응답 기다리지 않고 다른 작업 가능
• 지연성: 결과는 나중에 알 수 있음
• 독립성: 상대방 상태와 무관하게 요청 가능
```

**실생활 예시:**
- 이메일 전송: 보내고 바로 다른 일 가능
- 택배 발송: 맡기고 나면 끝, 배달은 택배사가 알아서
- 식당 진동벨: 벨 받고 자리에서 대기, 다른 일 가능

**프로그래밍 예시:**
- 메시지 큐 (Kafka, RabbitMQ)
- 이벤트 기반 아키텍처
- async/await 프로그래밍

### 비교표

| 구분 | 동기식 (Synchronous) | 비동기식 (Asynchronous) |
|------|---------------------|------------------------|
| 비유 | 전화 | 우편/택배 |
| 대기 | 응답까지 블로킹 | 논블로킹 (바로 다음 작업) |
| 결과 확인 | 즉시 | 나중에 (콜백/폴링) |
| 상대방 필요 | 반드시 필요 | 없어도 요청 가능 |
| 장애 영향 | 전파됨 | 격리됨 |
| 처리량 | 낮음 | 높음 |
| 사용 예 | 로그인, 결제 승인 | 알림, 로그 수집, 이메일 |

---
## 📚 Step 0: Flask가 뭐야?

### 한 줄 설명

> **Flask = 내 컴퓨터를 "전화 받을 수 있는 상태"로 만들어주는 도구**

```
일반 Python 파일     →    실행하면 끝, 아무도 연락 못함
Flask Python 파일    →    실행하면 대기, 누군가 연락하면 응답
```

### 용어 정리

| 비유 | 실제 용어 | 의미 |
|------|----------|------|
| 전화 받을 수 있는 상태 | **웹 서버**(Web Server) | 네트워크 요청을 기다리고 응답하는 프로그램 |
| 전화번호 | **URL / 엔드포인트** | 접속 주소 (예: `http://localhost:5000/chat`) |
| 전화가 오면 | **HTTP 요청**(Request) | 클라이언트가 서버에 데이터를 보내는 것 |
| 응답하면 | **HTTP 응답**(Response) | 서버가 클라이언트에게 결과를 돌려주는 것 |

### Flask의 정체

```
Flask = Python으로 만든 "마이크로 웹 프레임워크"

• 마이크로: 최소한의 기능만 제공 (가볍고 단순)
• 웹 프레임워크: 웹 서버를 쉽게 만들 수 있게 도와주는 도구 모음
```

> 📖 **공식 문서**: [Flask Documentation](https://flask.palletsprojects.com/) 참고

### 환경 준비: 가상환경 만들기

> 프로젝트마다 독립된 환경을 만들어 패키지 충돌을 방지합니다.

```bash
# 1) 프로젝트 폴더 생성 및 이동
mkdir flask-tutorial
cd flask-tutorial

# 2) 가상환경 생성
python -m venv venv

# 3) 가상환경 활성화
# Mac / Linux
source venv/bin/activate

# 4) 필요한 패키지 설치
pip install flask requests

# 5) 설치 확인
pip list
```

> 활성화되면 터미널 앞에 `(venv)`가 표시됩니다.
> ```
> (venv) flask-tutorial$
> ```

> 💡 **가상환경을 끄려면:** `deactivate` 입력

### 미니 실습: 가장 간단한 Flask

**1) 파일 작성 (hello.py)**

```python
# ============================================================
# Flask 기본 예제: 가장 간단한 웹 서버
# ============================================================

# Flask 라이브러리에서 Flask 클래스를 가져옴
# Flask: 웹 서버를 만들기 위한 핵심 클래스
from flask import Flask

# Flask 애플리케이션 인스턴스 생성
# __name__: 현재 모듈의 이름 (Python 내장 변수)
# Flask는 이 이름을 기준으로 템플릿, 정적 파일 등의 경로를 찾음
app = Flask(__name__)

# 라우트(Route) 데코레이터: URL 경로와 함수를 연결
# '/hello' 경로로 요청이 오면 아래 함수가 실행됨
# 데코레이터: 함수 위에 @로 시작하는 특수 문법, 함수에 기능을 추가
@app.route('/hello')
def say_hello():
    # 클라이언트에게 보낼 응답 (문자열 반환)
    # return 값이 HTTP 응답 본문(Body)이 됨
    return "안녕하세요!"

# Python 파일이 직접 실행될 때만 서버 시작
# 다른 파일에서 import될 때는 실행되지 않음
if __name__ == '__main__':
    # 웹 서버 실행
    # port=5000: 5000번 포트에서 요청 대기
    # 기본값: host='127.0.0.1' (로컬에서만 접근 가능)
    app.run(port=5000)
```

**2) 실행**

```bash
python hello.py
```

**3) 브라우저에서 확인**

```
http://localhost:5000/hello
```

화면에 "안녕하세요!"가 보이면 성공!

### 핵심 개념 정리

```python
@app.route('/hello')      # "/hello"로 전화 오면
def say_hello():          # 이 함수가 받아서
    return "안녕하세요!"    # 이 말을 해줌
```

| 비유 | Flask |
|------|-------|
| 전화번호 | `localhost:5000` |
| 내선번호 | `/hello` |
| 받는 사람 | `say_hello()` 함수 |

---
## 📚 Step 1: 두 프로그램이 "전화"하기

### 상황 설명

```
[손님 프로그램] ──전화──▶ [가게 프로그램]
   "사과 있어요?"           "네, 있어요!"
```

### 가게 프로그램 (shop.py)

```python
# ============================================================
# 가게 서버: HTTP 요청을 받아 재고를 확인해주는 서버
# 역할: "전화를 받는 쪽" (서버)
# ============================================================

from flask import Flask

# Flask 앱 생성
app = Flask(__name__)

# '/check' 엔드포인트 정의
# 손님이 이 URL로 요청하면 재고 확인 결과를 반환
@app.route('/check')
def check_item():
    # 서버 콘솔에 로그 출력 (디버깅용)
    # 실제로 요청이 왔는지 확인할 수 있음
    print("📞 전화 왔다! 재고 확인 중...")

    # 클라이언트에게 응답 반환
    # 이 문자열이 HTTP 응답으로 전송됨
    return "사과 있어요!"

# 이 파일을 직접 실행할 때만 서버 시작
if __name__ == '__main__':
    print("🏪 가게 오픈! 전화 기다리는 중...")
    # 5001번 포트에서 요청 대기
    # 서버는 종료하기 전까지 계속 실행 (대기 상태 유지)
    app.run(port=5001)
```

### 손님 프로그램 (customer.py)

```python
# ============================================================
# 손님 클라이언트: 가게 서버에 HTTP 요청을 보내는 프로그램
# 역할: "전화를 거는 쪽" (클라이언트)
# ============================================================

# requests: HTTP 요청을 보내는 라이브러리
# pip install requests 로 설치 필요
import requests

print("📱 가게에 전화 거는 중...")

# --------------------------------------------------------
# 동기식 HTTP 요청 (GET 방식)
# --------------------------------------------------------
# requests.get(): 지정한 URL로 GET 요청을 보내고 응답을 기다림
#
# 동작 순서:
# 1. localhost:5001 서버에 연결 시도
# 2. /check 경로로 GET 요청 전송
# 3. 서버가 응답할 때까지 대기 (블로킹!)
# 4. 응답을 받으면 response 객체에 저장
#
# 주의: 서버가 응답하지 않으면 이 줄에서 프로그램이 멈춤!
response = requests.get("http://localhost:5001/check")

# response.text: 서버가 보낸 응답 본문 (문자열)
# response.status_code: HTTP 상태 코드 (200=성공, 404=없음, 500=서버에러)
print(f"📱 가게 대답: {response.text}")
```

> `requests`는 다른 프로그램에게 HTTP 요청을 보내는 라이브러리입니다.
> **핵심**: `requests.get()`은 **동기식**으로 동작하여 응답이 올 때까지 기다립니다.

### 실습 순서

```bash
# 터미널 1: 가게 먼저 열기
python shop.py

# 터미널 2: 손님이 전화
python customer.py
```

**결과:**
```
# 터미널 1 (가게)
🏪 가게 오픈! 전화 기다리는 중...
📞 전화 왔다! 재고 확인 중...

# 터미널 2 (손님)
📱 가게에 전화 거는 중...
📱 가게 대답: 사과 있어요!
```

### 🔴 문제 체험: 가게가 문 닫으면?

```bash
# 터미널 1: 가게 종료 (Ctrl+C)

# 터미널 2: 손님이 전화
python customer.py
```

**결과: 에러!**
```
requests.exceptions.ConnectionError: 연결할 수 없음
```

> **핵심 깨달음:** 전화는 상대방이 받아야만 통화가 됨!

---
## 📚 Step 2: Docker로 옮기기

### 왜 Docker?

```
로컬 실행 문제:
• 터미널 여러 개 켜야 함
• 포트 충돌 날 수 있음
• 다른 컴퓨터에서 재현 어려움

Docker 사용 시:
• 각 프로그램이 독립된 방에서 실행
• 설정 파일 하나로 전체 관리
• 어디서든 동일하게 실행 가능
```

### 폴더 구조

```
phone-demo/
├── shop/
│   ├── app.py
│   └── Dockerfile
└── docker-compose.yml
```

### shop/app.py (아까와 거의 동일)

```python
# ============================================================
# Docker용 Flask 서버
# 로컬 버전과 다른 점: host='0.0.0.0' 설정
# ============================================================

from flask import Flask

app = Flask(__name__)

@app.route('/check')
def check_item():
    print("📞 전화 왔다!")
    return "사과 있어요!"

if __name__ == '__main__':
    # --------------------------------------------------------
    # host='0.0.0.0' 설정이 핵심!
    # --------------------------------------------------------
    # 127.0.0.1 (기본값): 같은 컴퓨터에서만 접근 가능
    # 0.0.0.0: 모든 네트워크 인터페이스에서 접근 허용
    #
    # Docker 컨테이너는 "별도의 컴퓨터"처럼 동작
    # → 127.0.0.1로 설정하면 컨테이너 외부에서 접근 불가!
    # → 0.0.0.0으로 설정해야 호스트에서 접근 가능
    #
    # 1교시에서 배운 내용 복습:
    # - 127.0.0.1 = 나 자신만 (localhost)
    # - 0.0.0.0 = 모든 곳에서 오는 연결 허용
    app.run(host='0.0.0.0', port=5001)
```

> ⚠️ **중요**: `host='0.0.0.0'`을 추가해야 Docker 외부에서 접근 가능 (1교시 복습!)

### shop/Dockerfile

```dockerfile
# ============================================================
# Dockerfile: 컨테이너 이미지를 만드는 설계도
# "이 컨테이너 안에 뭘 설치하고, 뭘 실행할지" 정의
# ============================================================

# 베이스 이미지 선택
# python:3.9-slim = Python 3.9이 설치된 경량 Linux
# slim 버전: 불필요한 패키지 제외하여 이미지 크기 최소화
FROM python:3.9-slim

# 작업 디렉토리 설정
# 컨테이너 내부에 /app 폴더를 만들고 이동
# 이후 명령어는 모두 이 폴더에서 실행됨
WORKDIR /app

# 패키지 설치
# RUN: 이미지 빌드 시 실행되는 명령어
# 컨테이너 안에 flask 라이브러리 설치
RUN pip install flask

# 파일 복사
# COPY <호스트 파일> <컨테이너 경로>
# 현재 폴더의 app.py를 컨테이너의 /app/으로 복사
# "." = 현재 WORKDIR (/app)
COPY app.py .

# 컨테이너 실행 시 수행할 명령어
# CMD: 컨테이너가 시작될 때 실행되는 기본 명령
# 배열 형식 ["명령어", "인자1", "인자2"]
CMD ["python", "app.py"]
```

### docker-compose.yml

```yaml
# ============================================================
# Docker Compose: 여러 컨테이너를 한번에 관리하는 설정 파일
# "어떤 서비스들을 어떻게 실행할지" 정의
# ============================================================

# Compose 파일 버전 (3.8 = 비교적 최신)
version: '3.8'

# 서비스 정의 섹션
# 각 서비스 = 하나의 컨테이너
services:
  # 서비스 이름: shop
  # 이 이름으로 다른 컨테이너에서 통신 가능 (DNS 역할)
  shop:
    # 빌드 설정: ./shop 폴더의 Dockerfile 사용
    build: ./shop

    # 포트 매핑: "호스트포트:컨테이너포트"
    # 호스트(내 컴퓨터)의 5001 포트로 들어오면
    # 컨테이너의 5001 포트로 전달
    ports:
      - "5001:5001"
```

### 실행 & 테스트

```bash
# 실행
docker-compose up --build

# 다른 터미널에서 테스트
curl http://localhost:5001/check
# 결과: 사과 있어요!
```

---
## 📚 Step 3: 전체 시스템 & 문제 체험

### 이제 만들 구조

```
[주문] ──전화1──▶ [재고]
  │
  ├───전화2──▶ [배송]  ← 느림 (3초)
  │
  └───전화3──▶ [알림]
```

실제 쇼핑몰처럼 주문이 들어오면:
1. 재고 확인
2. 배송 예약
3. 고객 알림

### 폴더 구조

```
phone-system/
├── docker-compose.yml
├── order/
│   ├── app.py
│   └── Dockerfile
├── inventory/
│   ├── app.py
│   └── Dockerfile
├── shipping/
│   ├── app.py
│   └── Dockerfile
└── notification/
    ├── app.py
    └── Dockerfile
```

### 각 서비스 코드

**inventory/app.py (재고)**

```python
# ============================================================
# 재고 서비스: 재고 확인 요청을 처리하는 API 서버
# 처리 시간: 약 0.5초 (빠른 편)
# ============================================================

from flask import Flask
import time  # time.sleep()으로 처리 시간 시뮬레이션

app = Flask(__name__)

# '/check' 엔드포인트: 재고 확인 API
@app.route('/check')
def check():
    print("📦 재고 확인 중...")

    # time.sleep(초): 지정한 시간만큼 프로그램 일시 정지
    # 실제로는 DB 조회, 외부 API 호출 등의 시간을 시뮬레이션
    time.sleep(0.5)  # 0.5초 대기 (처리 시간)

    return "재고 OK"

if __name__ == '__main__':
    # host='0.0.0.0': Docker 외부에서 접근 가능하도록
    # port=5001: 이 서비스는 5001번 포트 사용
    app.run(host='0.0.0.0', port=5001)
```

**shipping/app.py (배송) - ⚠️ 일부러 느림**

```python
# ============================================================
# 배송 서비스: 배송 예약 요청을 처리하는 API 서버
# ⚠️ 처리 시간: 3초 (일부러 느리게 설정!)
# → 동기식 통신의 문제점을 체험하기 위함
# ============================================================

from flask import Flask
import time

app = Flask(__name__)

# '/schedule' 엔드포인트: 배송 예약 API
@app.route('/schedule')
def schedule():
    print("🚚 배송 예약 중... (오래 걸림)")

    # ⚠️ 핵심: 3초나 걸리는 느린 처리!
    # 실제 상황: 외부 배송사 API 호출, 복잡한 경로 계산 등
    # 이 3초 동안 주문 서비스는 "대기" 상태 (블로킹)
    time.sleep(3)  # 3초 대기 - 동기식 문제의 핵심!

    return "배송 OK"

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5002)
```

**notification/app.py (알림)**

```python
# ============================================================
# 알림 서비스: 고객에게 알림을 발송하는 API 서버
# 처리 시간: 약 0.3초 (빠른 편)
# ============================================================

from flask import Flask
import time

app = Flask(__name__)

# '/send' 엔드포인트: 알림 발송 API
@app.route('/send')
def send():
    print("📱 알림 발송!")

    # 알림 발송 처리 시간 (SMS, 푸시 알림 등)
    time.sleep(0.3)

    return "알림 OK"

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5003)
```

### order/app.py (주문) - 핵심!

```python
# ============================================================
# 주문 서비스: 전체 주문 프로세스를 처리하는 핵심 서비스
#
# 동기식 통신의 문제점을 직접 보여주는 코드!
# - 재고 → 배송 → 알림 순서로 "차례로" 호출
# - 각 서비스의 응답을 기다림 (블로킹)
# - 하나라도 실패하면 전체 실패
# ============================================================

from flask import Flask
import requests  # HTTP 요청을 보내기 위한 라이브러리
import time      # 시간 측정을 위해 사용

app = Flask(__name__)

# '/order' 엔드포인트: 주문 생성 API
@app.route('/order')
def create_order():
    # ========================================================
    # 시간 측정 시작
    # time.time(): 현재 시간을 초 단위로 반환 (Unix timestamp)
    # 나중에 (현재시간 - start)로 경과 시간 계산
    # ========================================================
    start = time.time()
    print("\n" + "="*40)
    print("🛒 주문 접수!")
    print("="*40)

    # ========================================================
    # 전화 1: 재고 확인 (동기식 호출)
    # ========================================================
    print("📞 재고에 전화...")
    try:
        # requests.get(): 동기식 HTTP GET 요청
        #
        # URL 분석: "http://inventory:5001/check"
        # - inventory: Docker Compose 서비스 이름 (DNS로 IP 자동 변환)
        # - 5001: 재고 서비스 포트
        # - /check: 재고 확인 엔드포인트
        #
        # timeout=5: 5초 안에 응답 없으면 예외 발생
        # → 무한 대기 방지 (운영 환경에서 필수!)
        r = requests.get("http://inventory:5001/check", timeout=5)
        print(f"   → {r.text}")  # 응답 내용 출력
    except:
        # 연결 실패 시 (서비스 다운, 네트워크 오류, 타임아웃 등)
        # 500 = HTTP 상태 코드 "Internal Server Error"
        return "❌ 재고 서비스 연결 실패", 500

    # ========================================================
    # 전화 2: 배송 예약 (동기식 호출) - ⚠️ 여기서 3초 대기!
    # ========================================================
    print("📞 배송에 전화...")
    try:
        # ⚠️ 핵심: 이 줄에서 3초간 "멈춤" (블로킹)
        # shipping 서비스가 time.sleep(3)으로 3초 대기하기 때문
        #
        # 문제점:
        # 1. 주문 서비스는 아무것도 못하고 기다림
        # 2. 클라이언트(사용자)도 3초간 응답을 못 받음
        # 3. 다른 주문 요청도 이 요청이 끝날 때까지 밀림
        #
        # timeout=10: 배송이 느린 걸 알기에 더 긴 타임아웃 설정
        r = requests.get("http://shipping:5002/schedule", timeout=10)
        print(f"   → {r.text}")
    except:
        # ⚠️ 문제: 재고 확인은 성공했는데 배송 실패!
        # → 이미 재고 차감했다면? 롤백이 필요하지만 여기선 처리 안 됨
        return "❌ 배송 서비스 연결 실패", 500

    # ========================================================
    # 전화 3: 알림 발송 (동기식 호출)
    # ========================================================
    print("📞 알림에 전화...")
    try:
        r = requests.get("http://notification:5003/send", timeout=5)
        print(f"   → {r.text}")
    except:
        # ⚠️ 문제: 재고 확인 + 배송 예약 성공했는데 알림 실패!
        # → 알림은 실패해도 주문은 성공 처리해야 하지 않을까?
        # → 동기식에서는 이런 부분적 성공 처리가 어려움
        return "❌ 알림 서비스 연결 실패", 500

    # ========================================================
    # 시간 측정 종료 및 결과 반환
    # ========================================================
    elapsed = time.time() - start  # 경과 시간 계산
    print(f"\n⏱️ 총 {elapsed:.1f}초 걸림")
    print("="*40 + "\n")

    # 모든 서비스 호출 성공 시에만 여기 도달
    # 예상 시간: 0.5초(재고) + 3초(배송) + 0.3초(알림) ≈ 4초
    return f"✅ 주문 완료! ({elapsed:.1f}초)"

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
```

> **핵심**: 서비스 이름(`inventory`, `shipping`, `notification`)으로 통신! (1교시 복습)
>
> **동기식 문제 요약**:
> - 순차 실행: 재고 → 배송 → 알림 (병렬 X)
> - 대기 시간 누적: 0.5 + 3 + 0.3 = 약 4초
> - 하나 실패 시 전체 실패 (부분 성공 처리 어려움)

### Dockerfile (4개 모두 동일)

```dockerfile
# 모든 서비스가 동일한 Dockerfile 구조 사용
FROM python:3.9-slim
WORKDIR /app
# flask: 웹 서버 프레임워크
# requests: HTTP 클라이언트 (order 서비스에서 사용)
RUN pip install flask requests
COPY app.py .
CMD ["python", "app.py"]
```

### docker-compose.yml

```yaml
# ============================================================
# Docker Compose: 4개 서비스를 한번에 관리
#
# 구조:
# [order] ─→ [inventory]  (재고 확인)
#    │
#    ├─→ [shipping]       (배송 예약)
#    │
#    └─→ [notification]   (알림 발송)
# ============================================================

version: '3.8'

services:
  # --------------------------------------------------------
  # 주문 서비스 (Entry Point - 외부에서 접근 가능)
  # --------------------------------------------------------
  order:
    build: ./order
    # ⚠️ 포트 매핑: 이 서비스만 외부 노출
    # 호스트에서 curl http://localhost:5000/order 로 접근 가능
    ports:
      - "5000:5000"
    # 참고: depends_on을 추가하면 다른 서비스 시작 후 실행 가능
    # depends_on:
    #   - inventory
    #   - shipping
    #   - notification

  # --------------------------------------------------------
  # 재고 서비스 (내부 전용 - 포트 매핑 없음)
  # --------------------------------------------------------
  inventory:
    build: ./inventory
    # 포트 매핑 없음 = 외부에서 직접 접근 불가
    # Docker 네트워크 내에서만 "inventory:5001"로 접근

  # --------------------------------------------------------
  # 배송 서비스 (내부 전용 - 의도적으로 느리게 설정)
  # --------------------------------------------------------
  shipping:
    build: ./shipping
    # 이 서비스가 3초 딜레이로 전체 시스템을 느리게 만듦

  # --------------------------------------------------------
  # 알림 서비스 (내부 전용)
  # --------------------------------------------------------
  notification:
    build: ./notification
```

**핵심 포인트:**

| 항목 | 설명 |
|------|------|
| **서비스 이름 통신** | `http://shipping:5002` - 서비스 이름이 DNS 역할 |
| **외부 포트** | `order`만 포트 매핑 (5000:5000) |
| **내부 서비스** | inventory, shipping, notification은 포트 매핑 없음 |
| **네트워크** | Docker Compose가 자동으로 브릿지 네트워크 생성 |

---
## 🧪 실습 시나리오

### 실습 A: 정상 동작

```bash
docker-compose up --build

# 다른 터미널
curl http://localhost:5000/order
```

**관찰:** 약 4초 걸림 (0.5 + 3 + 0.3)

```
🛒 주문 접수!
📞 재고에 전화...
   → 재고 OK
📞 배송에 전화...
   → 배송 OK        ← 여기서 3초 대기!
📞 알림에 전화...
   → 알림 OK
⏱️ 총 3.9초 걸림
```

> **문제 1:** 배송이 느리면 전체 주문이 느려짐!

#### 📋 실제 실행 결과 (order 서비스 로그)

```
========================================
🛒 주문 접수!
========================================
📞 재고에 전화...
   → 재고 OK
📞 배송에 전화...
   → 배송 OK
📞 알림에 전화...
   → 알림 OK

⏱️ 총 3.9초 걸림
========================================
192.168.97.1 - - [11/Jan/2026 15:39:38] "GET /order HTTP/1.1" 200 -
```

**로그 해석:**
| 로그 항목 | 의미 |
|----------|------|
| `192.168.97.1` | 요청을 보낸 클라이언트 IP (Docker 호스트) |
| `GET /order HTTP/1.1` | HTTP GET 메서드로 /order 경로 요청 |
| `200` | HTTP 상태 코드 - 성공 |
| `-` | 응답 크기 (여기선 표시 안됨) |

#### 📋 각 서비스별 로그

```bash
# inventory 서비스
📦 재고 확인 중...
192.168.97.3 - - [11/Jan/2026 15:39:35] "GET /check HTTP/1.1" 200 -

# shipping 서비스 (3초 대기)
🚚 배송 예약 중... (오래 걸림)
192.168.97.3 - - [11/Jan/2026 15:39:38] "GET /schedule HTTP/1.1" 200 -

# notification 서비스
📱 알림 발송!
192.168.97.3 - - [11/Jan/2026 15:39:38] "GET /send HTTP/1.1" 200 -
```

**핵심 관찰:**
- inventory 요청 시간: `15:39:35`
- shipping/notification 완료 시간: `15:39:38`
- 약 3초 차이 → **shipping 서비스의 `time.sleep(3)`이 전체 응답 시간을 지연**

### 실습 B: 배송 서비스 죽이기

```bash
docker-compose stop shipping

curl http://localhost:5000/order
```

**결과:** "❌ 배송 서비스 연결 실패"

```
🛒 주문 접수!
📞 재고에 전화...
   → 재고 OK
📞 배송에 전화...
❌ 배송 서비스 연결 실패    ← 전체 주문 실패!
```

> **문제 2:** 하나가 죽으면 전체가 실패!

#### 📋 실제 실행 결과 (order 서비스 로그)

```
========================================
🛒 주문 접수!
========================================
📞 재고에 전화...
   → 재고 OK
📞 배송에 전화...
192.168.97.1 - - [11/Jan/2026 15:40:32] "GET /order HTTP/1.1" 500 -
```

**로그 해석:**
| 로그 항목 | 의미 |
|----------|------|
| `500` | HTTP 상태 코드 - **Internal Server Error** |
| 재고 OK 후 멈춤 | 배송 서비스 연결 실패로 인해 진행 불가 |

**핵심 관찰:**
- 재고 확인(`inventory`)은 **성공** → `재고 OK` 출력
- 배송 예약(`shipping`) 시도 시 **연결 실패** → 전체 요청 500 에러
- **부분 성공 문제**: 재고 차감 로직이 있었다면 롤백이 필요하지만, 동기식에서는 처리 어려움

### 실습 C: 다시 살리기

```bash
docker-compose start shipping

curl http://localhost:5000/order
```

#### 📋 실제 실행 결과

```bash
# 응답
✅ 주문 완료! (3.9초)
```

> **문제 3:** 그 사이 들어온 주문은? **사라짐!**

**핵심 관찰:**
- 서비스 복구 후 **새 요청은 정상 처리**됨
- 하지만 **장애 중에 실패한 요청은 어디에도 기록되지 않음**
- 동기식 통신은 **데이터를 저장하지 않음** (휘발성)
- Kafka 같은 메시지 큐가 있었다면? → 장애 복구 후 큐에서 메시지를 가져와 처리 가능

---
## 📋 정리: 전화 방식(동기식 통신)의 문제

| 체험한 것 | 문제 | 실제 상황 |
|----------|------|----------|
| 4초 대기 | 느린 서비스가 전체를 느리게 | 배송사 API가 느리면 결제 화면이 멈춤 |
| 서비스 중지 시 실패 | 하나 죽으면 전체 실패 | 알림 서버 장애 → 주문 불가 |
| 주문 유실 | 대기 중인 요청이 사라짐 | 장애 복구 후 주문 데이터 없음 |

### 동기식 통신의 근본적 한계

```
동기식 = "전화"
─────────────────────────────────────────────────────
• 상대방이 받아야만 통화 가능
• 통화 중에는 다른 일 못함 (대기)
• 상대방이 없으면 연결 불가
• 통화 내용을 저장하지 않음 (휘발성)
```

---
## 🎯 다음 시간 예고: Kafka 도입

> "중간에 **우체국**이 있으면 어떨까?"

```
현재 (전화 = 동기식):
┌────────┐              ┌────────┐
│  주문   │ ───직접───▶   │  배송    │  ← 안 받으면 실패!
└────────┘              └────────┘

Kafka 도입 (우체국 = 비동기식):
┌────────┐    ┌────────┐    ┌────────┐
│  주문   │ ─▶ │ 우체국   │ ─▶ │  배송   │  ← 편지 맡기면 끝!
└────────┘    │(Kafka) │    └────────┘    나중에 가져감
              └────────┘
```

### Kafka가 해결하는 것

| 문제 | 해결 |
|------|------|
| 느림 | 편지 맡기고 바로 다음 일 (비동기) |
| 장애 전파 | 우체국이 보관, 나중에 배달 |
| 데이터 유실 | 편지가 우체국에 저장됨 (영속성) |

---
## ❓ FAQ

**Q1. 동기식 통신이 항상 나쁜 건가요?**

아닙니다. 즉각적인 응답이 필요한 경우(예: 결제 승인, 로그인)에는 동기식이 적합합니다.
비동기가 필요한 경우: 알림 발송, 로그 수집, 이메일 전송 등 "나중에 처리해도 되는" 작업

---

**Q2. requests.get()에서 timeout은 왜 설정하나요?**

timeout 없이 요청하면 상대방이 응답하지 않을 때 영원히 대기합니다.
운영 환경에서는 반드시 timeout을 설정해야 합니다.

---

**Q3. 서비스 간 통신에서 HTTP 대신 다른 방법은 없나요?**

있습니다:
- **gRPC**: 더 빠른 바이너리 프로토콜
- **메시지 큐**: Kafka, RabbitMQ (비동기)
- **GraphQL**: 필요한 데이터만 요청

---

**Q4. Docker Compose에서 서비스 이름으로 통신이 안 되면?**

같은 `docker-compose.yml`에 정의되어 있는지 확인하세요.
서비스가 완전히 시작되기 전에 요청하면 실패할 수 있습니다 (`depends_on` 사용).

---

**Q5. 실제 운영 환경에서도 이렇게 단순하게 구성하나요?**

실제로는 로드밸런서, 서비스 디스커버리, 헬스체크, 재시도 로직 등이 추가됩니다.
이 실습은 문제점을 체험하기 위한 단순화된 버전입니다.

---
## 📝 퀴즈

### Q1. Flask에서 `app.run(host='0.0.0.0')`으로 설정하는 이유는?

- A) 속도를 높이기 위해
- B) 보안을 강화하기 위해
- C) Docker 컨테이너 외부에서 접근 가능하게 하기 위해
- D) 포트 충돌을 방지하기 위해

<details>
<summary>정답 보기</summary>

**정답: C**

`127.0.0.1`은 로컬에서만 접근 가능합니다.
Docker 컨테이너는 별도 네트워크이므로, 외부 접근을 위해 `0.0.0.0`이 필요합니다.
</details>

---

### Q2. 동기식 통신에서 하위 서비스가 3초 걸리면 전체 응답 시간은?

- A) 하위 서비스 응답 시간과 무관
- B) 최소 3초 이상
- C) 정확히 3초
- D) 3초보다 짧음

<details>
<summary>정답 보기</summary>

**정답: B**

동기식 통신은 하위 서비스의 응답을 기다립니다.
따라서 전체 응답 시간은 최소 3초 + 다른 처리 시간입니다.
</details>

---

### Q3. 동기식 통신에서 하위 서비스가 장애일 때 발생하는 문제는?

- A) 자동으로 재시도됨
- B) 다른 서비스로 우회됨
- C) 요청이 큐에 저장됨
- D) 전체 요청이 실패함

<details>
<summary>정답 보기</summary>

**정답: D**

동기식 통신은 상대방이 응답해야 진행됩니다.
하위 서비스 장애 시 연결 실패로 전체 요청이 실패합니다.
</details>

---

### Q4. Docker Compose에서 서비스 간 통신 시 사용하는 주소는?

- A) localhost:포트
- B) 127.0.0.1:포트
- C) 서비스이름:포트
- D) 컨테이너IP:포트

<details>
<summary>정답 보기</summary>

**정답: C**

Docker Compose는 서비스 이름을 DNS로 등록합니다.
`http://inventory:5001`처럼 서비스 이름으로 통신합니다.
</details>

---
## 📋 핵심 요약

| 개념 | 설명 |
|------|------|
| **Flask** | Python 마이크로 웹 프레임워크 |
| **동기식 통신** | 요청 → 대기 → 응답 (전화 방식) |
| **문제점** | 느림, 장애 전파, 데이터 유실 |
| **host='0.0.0.0'** | 모든 네트워크에서 접근 허용 |
| **서비스 이름 통신** | Docker Compose 내부 DNS |

### 동기식 vs 비동기식 (예고)

| 구분 | 동기식 (전화) | 비동기식 (우편) |
|------|-------------|----------------|
| 대기 | 응답까지 대기 | 보내고 바로 다음 일 |
| 장애 | 연결 실패 → 전체 실패 | 우체국이 보관 |
| 데이터 | 휘발성 | 영속성 (저장됨) |
| 예시 | HTTP 요청 | Kafka, RabbitMQ |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| Flask로 간단한 웹 서버를 만들 수 있나요? | ☐ |
| `host='0.0.0.0'`이 왜 필요한지 설명할 수 있나요? | ☐ |
| 동기식 통신의 3가지 문제점을 말할 수 있나요? | ☐ |
| Docker Compose에서 서비스 이름으로 통신하는 방법을 아나요? | ☐ |
| Kafka가 왜 필요한지 한 문장으로 설명할 수 있나요? | ☐ |

---


# Day 06 - 3교시: Kafka 개념 소개
> Apache Kafka 핵심 개념 완벽 가이드

## 🎯 학습 목표
이 파트를 마치면 다음을 이해할 수 있습니다:

- Kafka가 무엇이고 왜 필요한지 설명할 수 있다
- Kafka의 핵심 용어 (Topic, Partition, Producer, Consumer, Broker)를 이해한다
- Consumer Group과 Offset의 개념을 설명할 수 있다
- Topic Replication이 왜 중요한지 이해한다
- Zookeeper와 KRaft 모드의 차이를 알 수 있다

---

## 📚 2교시 복습: 동기식 통신의 문제점

```
현재 (전화 = 동기식):
┌────────┐              ┌────────┐
│  주문   │ ───직접───▶   │  배송   │  ← 안 받으면 실패!
└────────┘              └────────┘

문제점:
1. 느림: 배송이 3초 걸리면 주문도 3초 대기
2. 장애 전파: 배송 서버 다운 → 주문 전체 실패
3. 데이터 유실: 장애 중 요청은 사라짐
```

> 💡 **핵심 깨달음**: "중간에 우체국이 있으면 어떨까?"

---
## 🚀 Kafka란?

### 한 줄 정의

> **Apache Kafka**: 실시간 데이터 파이프라인과 스트리밍 애플리케이션을 구축하기 위한
> **분산 이벤트 스트리밍 플랫폼** (Distributed Event Streaming Platform)

- **LinkedIn**에서 2011년 개발, 현재 Apache 재단 오픈소스
- 전 세계 80% 이상의 Fortune 100 기업이 사용
- 하루 **수조 건**의 메시지 처리 가능

### Kafka Cluster 구조

![Kafka Cluster with 4 Topics](https://mintcdn.com/conduktor/czPJDOcm9Lmdmfbk/learn/images/Apache_Kafka_Cluster_with_4_topics.png?w=1650&fit=max&auto=format&n=czPJDOcm9Lmdmfbk&q=85&s=3f2c8d908980886fddb4533dc82fb361)

> 📖 **참고 문서**: [Conduktor Kafka Learn](https://docs.conduktor.io/learn/fundamentals/)

---
## 📬 Topic (토픽)

### Topic이란?

> **Topic = 메시지가 저장되는 "주제별 우편함"**

Kafka는 Topic을 사용하여 관련 메시지를 구성합니다.
데이터베이스에서 테이블을 사용하는 것과 유사합니다.

- 예: `logs` 토픽 → 애플리케이션 로그 메시지
- 예: `purchases` 토픽 → 구매 데이터
- 예: `orders` 토픽 → 주문 이벤트

### Topic의 특징

| 특징 | 설명 |
|------|------|
| **다양한 포맷** | JSON, Avro, Protobuf 등 어떤 형식이든 저장 가능 |
| **쿼리 불가** | SELECT, JOIN 같은 쿼리 ❌ → Producer/Consumer로만 접근 |
| **자동 삭제** | 기본 7일 후 자동 삭제 (설정 가능) |
| **불변성** | 한번 쓰인 데이터는 수정/삭제 불가 |

---
## 📦 Partition (파티션)

### Partition이란?

> **Partition = Topic을 쪼갠 조각들**

Topic은 **Partition**이라는 더 작은 단위로 나뉩니다.
하나의 Topic은 보통 여러 개의 Partition을 가집니다 (실무에서는 100개 이상도 흔함).

![Kafka Topics with 3 Partitions](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Topics_1.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=3e42e5687d033d8d4d8b297a4b04c880)

### Partition 번호 체계

- Partition은 **0번부터** 시작합니다
- 3개의 Partition이 있다면: `Partition 0`, `Partition 1`, `Partition 2`

### 왜 Partition으로 나누나요?

| 이유 | 설명 |
|------|------|
| **병렬 처리** | 여러 Consumer가 동시에 읽을 수 있음 |
| **확장성** | Partition을 늘려 처리량 증가 |
| **내결함성** | 일부 장애에도 다른 Partition은 정상 동작 |

---
## 🔢 Offset (오프셋)

### Offset이란?

> **Offset = 메시지의 고유 위치 번호**

Kafka는 메시지가 Partition에 기록될 때 **offset**이라는 정수 값을 부여합니다.

```
Partition 0:
┌────┬────┬────┬────┬────┬────┐
│ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │  ← offset
└────┴────┴────┴────┴────┴────┘
  ↑                          ↑
 가장 오래된 메시지      가장 최신 메시지
```

### Offset의 특징

| 특징 | 설명 |
|------|------|
| **Partition별 독립** | 각 Partition은 자체 offset 번호를 가짐 |
| **0부터 시작** | 첫 메시지는 offset 0 |
| **무한 증가** | 재사용되지 않고 계속 증가 |
| **순서 보장** | **Partition 내에서만** 순서 보장 |

> ⚠️ **중요**: Kafka는 Partition 내에서는 순서를 보장하지만,
> **여러 Partition 간에는 순서를 보장하지 않습니다!**

---
## 🏭 Producer (프로듀서)

### Producer란?

> **Producer = 메시지를 Kafka로 보내는 애플리케이션**

![Kafka Producer](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Producers_1.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=17ff0279c9e4de47176847cca6ed8e4d)

Producer는 클라이언트 라이브러리를 사용하여 Kafka에 데이터를 씁니다.
Python, Java, Go 등 다양한 언어를 지원합니다.

### Message Key

Producer는 메시지에 **Key**를 지정할 수 있습니다.

![Kafka Producer with Key](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Producers_2.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=8d326376928b91ffd93b4aedef1d861b)

| Key 상태 | 동작 |
|----------|------|
| **Key = null** | Round-robin 방식으로 Partition에 분배 |
| **Key 지정** | 같은 Key는 **항상 같은 Partition**으로 전송 |

> 💡 **예시**: 물류 회사에서 `truck_id`를 Key로 사용하면,
> 한 트럭의 모든 GPS 데이터가 같은 Partition에 저장되어 순서가 보장됩니다.

### Message 구조

![Kafka Message Anatomy](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Producers_3.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=572b423d4a502b8d8fb4f4cbd00633df)

| 구성 요소 | 설명 |
|-----------|------|
| **Key** | 선택 사항, null 가능 |
| **Value** | 메시지 본문 (실제 데이터) |
| **Compression** | 압축 방식 (none, gzip, snappy, lz4, zstd) |
| **Headers** | 메타데이터용 key-value 쌍 |
| **Partition + Offset** | 메시지의 고유 식별자 |
| **Timestamp** | 생성 시각 |

---
## 🔄 Serialization (직렬화)

### 직렬화란?

> **Serialization = 객체를 바이트 배열로 변환하는 과정**

Kafka Broker는 Key와 Value를 **바이트 배열**로만 받습니다.
따라서 Producer는 데이터를 직렬화해서 보내야 합니다.

![Kafka Serialization](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Producers_4.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=ec8a69e5c3eb75a730dec4574fb7c8f6)

### 기본 제공 Serializer

| 데이터 타입 | Serializer |
|-------------|------------|
| String | StringSerializer |
| Integer | IntegerSerializer |
| Float | FloatSerializer |

### 고급 포맷

- **JSON Schema**
- **Apache Avro**
- **Protobuf**

> 이러한 포맷들은 Confluent Schema Registry와 함께 사용됩니다.

---
## 🎯 Key Hashing (키 해싱)

![Hash Function](https://upload.wikimedia.org/wikipedia/commons/thumb/2/2b/Cryptographic_Hash_Function.svg/960px-Cryptographic_Hash_Function.svg.png)

### Partitioner란?

> **Partitioner = 메시지가 어느 Partition으로 갈지 결정하는 로직**

![Kafka Key Hashing](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Producers_5.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=c05ddfdf890c1ef445e92bdbf89dff2d)

### 기본 알고리즘: murmur2

```python
# Kafka 기본 파티션 결정 공식
target_partition = abs(murmur2(key_bytes)) % num_partitions
```

- 같은 Key는 항상 같은 결과 → **같은 Partition**
- Key가 null이면 Round-robin 방식

> 💡 기본 Partitioner는 `partitioner.class` 설정으로 변경 가능하지만,
> 특별한 이유 없이 변경하는 것은 권장하지 않습니다.

---
## 📥 Consumer (컨슈머)

### Consumer란?

> **Consumer = Kafka에서 메시지를 가져오는 애플리케이션**

![Kafka Consumer](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Consumers_1.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=94ce11086ddb218167213622a9ab065f)

Consumer는 클라이언트 라이브러리를 사용하여 하나 이상의 Topic에서 데이터를 읽습니다.

### 읽기 방식

| 특징 | 설명 |
|------|------|
| **순차 읽기** | 낮은 offset → 높은 offset 순으로 읽음 |
| **뒤로 가기 불가** | 이미 읽은 데이터로 돌아갈 수 없음 (offset 조정으로만 가능) |
| **Partition 내 순서** | 각 Partition 내에서만 순서 보장 |
| **Pull 모델** | Broker가 밀어주는 게 아니라, Consumer가 당겨옴 |

### 기본 동작

> ⚠️ **주의**: Consumer는 기본적으로 **연결 이후의 새 메시지만** 읽습니다.
> 과거 데이터를 읽으려면 명시적으로 설정해야 합니다.

---
## 🔄 Deserialization (역직렬화)

### 역직렬화란?

> **Deserialization = 바이트 배열을 객체로 변환하는 과정**

Consumer가 받은 데이터는 **직렬화된 형식과 동일하게** 역직렬화해야 합니다.

![Kafka Deserialization](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Consumers_2.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=5698cfc7b4346195081270e78040785b)

### Serializer ↔ Deserializer 매칭

| 직렬화 | 역직렬화 |
|--------|----------|
| StringSerializer | StringDeserializer |
| IntegerSerializer | IntegerDeserializer |
| AvroSerializer | AvroDeserializer |

### Poison Pill (독약 메시지)

> ⚠️ **주의**: 약속된 직렬화 형식과 다른 메시지는 "Poison Pill"이 됩니다.
> 이런 메시지는 Consumer에서 처리 실패를 일으키고 디버깅이 어렵습니다.

**권장 사항**: Topic 생성 후 직렬화 형식을 **절대 변경하지 마세요**.
형식 변경이 필요하면 새 Topic을 만들고 애플리케이션을 마이그레이션하세요.

---
## 👥 Consumer Group (컨슈머 그룹)

### Consumer Group이란?

> **Consumer Group = 같은 역할을 하는 Consumer들의 모임**

같은 애플리케이션에 속한 Consumer들은 그룹으로 묶일 수 있습니다.

![Consumer Group reading from topic with 5 partitions](https://mintcdn.com/conduktor/czPJDOcm9Lmdmfbk/learn/images/Consumer_Group_reading_from_topic_with_5_partitions.png)

### Group ID 설정

```python
# Consumer 설정 시 group.id 지정
consumer = KafkaConsumer(
    'my-topic',
    group_id='my-application-group'  # 같은 앱은 같은 group.id 사용
)
```

### Partition 할당 규칙

![Kafka Consumer Groups](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Consumer_Groups_1.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=013ba41c1e216895f9d5c806b3bb843e)

| 규칙 | 설명 |
|------|------|
| **1 Partition : 1 Consumer** | 각 Partition은 그룹 내 **하나의 Consumer만** 읽음 |
| **1 Consumer : N Partitions** | 하나의 Consumer는 **여러 Partition**을 읽을 수 있음 |

![Consumer Group Partition Assignment](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Consumer_Groups_2.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=13edfece53370a245fdba4e5938701b5)

### Consumer 수 vs Partition 수

- Consumer 수 < Partition 수 → 일부 Consumer가 여러 Partition 담당
- Consumer 수 = Partition 수 → 1:1 매핑 (이상적)
- Consumer 수 > Partition 수 → **일부 Consumer는 놀게 됨!**

---
## 📍 Consumer Offset

### Consumer Offset이란?

> **Consumer Offset = Consumer가 마지막으로 읽은 메시지의 위치**

Kafka Broker는 `__consumer_offsets`라는 **내부 Topic**에
각 Consumer Group이 마지막으로 처리한 메시지의 위치를 기록합니다.

![Consumer Offset Commit](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Consumer_Groups_3_2x.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=22659a1d48fa2ffb4f1689008f478740)

### Offset Commit이 필요한 이유

| 상황 | Offset 활용 |
|------|-------------|
| **Consumer 재시작** | 마지막 위치부터 다시 읽기 |
| **Consumer 장애** | 다른 Consumer가 이어받아 처리 |
| **리밸런싱** | Partition 재할당 시 진행 상황 유지 |

> 💡 대부분의 클라이언트 라이브러리는 **자동으로** offset을 주기적으로 커밋합니다.

### Delivery Semantics (전달 보장 수준)

| 방식 | 설명 | 특징 |
|------|------|------|
| **At Most Once** | 받자마자 커밋 | 메시지 손실 가능 |
| **At Least Once** | 처리 후 커밋 (✅ 권장) | 중복 가능 |
| **Exactly Once** | 트랜잭션 사용 | 정확히 한 번만 |

---
## 🖥️ Broker (브로커)

### Broker란?

> **Broker = Kafka 서버 하나**

Kafka Broker는 JVM(Java 11+) 위에서 실행되는 프로그램입니다.

![Kafka Brokers](https://mintcdn.com/conduktor/czPJDOcm9Lmdmfbk/learn/images/Kafka_Brokers_1.png?w=1650&fit=max&auto=format&n=czPJDOcm9Lmdmfbk&q=85&s=b51118c93f49cff2423fa24b7c716515)

### Kafka Cluster

> **Cluster = 여러 Broker의 집합**

- 단일 Broker부터 수백, 수천 대까지 확장 가능
- 각 Broker는 고유한 **숫자 ID**를 가짐

### Broker와 Partition 분배

Broker는 서버 디렉토리에 데이터를 저장하며,
각 Topic-Partition은 자체 하위 디렉토리를 갖습니다.
**Partition은 Broker들에 고르게 분배**됩니다.

![Kafka Broker Partition Distribution](https://mintcdn.com/conduktor/czPJDOcm9Lmdmfbk/learn/images/Kafka_Brokers_2.png?w=1650&fit=max&auto=format&n=czPJDOcm9Lmdmfbk&q=85&s=4f0eb70253e1240c00662d21aa0284d7)

### Bootstrap Server

> 클라이언트가 Kafka 클러스터에 연결할 때 사용하는 초기 Broker

![Kafka Bootstrap Server](https://mintcdn.com/conduktor/czPJDOcm9Lmdmfbk/learn/images/Kafka_Brokers_3.png?w=1650&fit=max&auto=format&n=czPJDOcm9Lmdmfbk&q=85&s=0161f314c91f07a4d22016ef31b00605)

- **어떤 Broker든** Bootstrap Server로 사용 가능
- 연결 시 클러스터의 **모든 Broker 정보**를 제공받음
- 보통 **2~3개**의 Bootstrap Server를 지정 (장애 대비)

---
## 🔄 Topic Replication (복제)

### Replication이란?

> **Replication = 같은 데이터를 여러 Broker에 복사하는 것**

Kafka 복제는 데이터를 여러 Broker에 기록하여 **데이터 손실을 방지**합니다.
이를 통해 Broker 장애가 발생해도 데이터 가용성을 유지합니다.

![Kafka Topic Replication](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Topic_Replication_1.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=3f14440abab538fac4163dd624e05cc0)

### Replication Factor

| Factor | 설명 | 사용 상황 |
|--------|------|----------|
| **1** | 복제 없음 | 개발 환경만 |
| **2** | 1개 백업 | 최소한의 내결함성 |
| **3** | 2개 백업 (✅ 권장) | 프로덕션 표준 |

> 💡 **예시**: 3개 Broker 클러스터에서 Replication Factor 2로 설정하면,
> Broker 101에 쓴 데이터가 자동으로 Broker 102에도 복사됩니다.

### Leader와 Replica

| 역할 | 설명 |
|------|------|
| **Leader** | 모든 읽기/쓰기를 담당하는 주 Broker |
| **Replica** | Leader의 복사본을 가진 Broker |

### ISR (In-Sync Replica)

> **ISR = Leader와 동기화된 Replica들**

![Kafka ISR](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Topic_Replication_3.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=e2a80380f54113643841108447690538)

- Leader 장애 시, **ISR 중 하나**가 새 Leader로 선출됨
- 동기화가 늦은 Replica는 ISR에서 제외됨

---
## ✅ Producer Acknowledgments (acks)

Producer가 메시지 전송 후 얼마나 기다릴지 설정합니다.

### acks=0 (Fire and Forget)

![acks=0](https://mintcdn.com/conduktor/czPJDOcm9Lmdmfbk/learn/images/Adv_Producer_Acks_DD_1.png?w=1650&fit=max&auto=format&n=czPJDOcm9Lmdmfbk&q=85&s=d793f7b4ac06da620137cef17788b277)

- 응답을 기다리지 않음
- **최고 속도**, 데이터 손실 위험

### acks=1 (Leader Only)

![acks=1](https://mintcdn.com/conduktor/czPJDOcm9Lmdmfbk/learn/images/Adv_Producer_Acks_DD_2.png?w=1650&fit=max&auto=format&n=czPJDOcm9Lmdmfbk&q=85&s=c5697e6a9664790c820248185fa19560)

- Leader가 받으면 완료
- Leader 장애 시 데이터 손실 가능

### acks=all (All ISRs)

![acks=all](https://mintcdn.com/conduktor/czPJDOcm9Lmdmfbk/learn/images/Adv_Producer_Acks_DD_3.png?w=1650&fit=max&auto=format&n=czPJDOcm9Lmdmfbk&q=85&s=bb2af6cbd69546efc384dd66f304fef6)

- 모든 ISR이 복제 완료해야 성공
- **가장 안전**, 속도는 느림 (✅ 권장)

| acks | 속도 | 안전성 | 사용 상황 |
|------|------|--------|----------|
| 0 | 최고 | 낮음 | 로그, 메트릭 (손실 허용) |
| 1 | 중간 | 중간 | 일반적인 경우 |
| all | 낮음 | 최고 | 금융, 주문 등 중요 데이터 |

---
## 🦁 Zookeeper

### Zookeeper란?

> **Zookeeper = Kafka 클러스터의 상태를 관리하는 외부 시스템**

![Zookeeper with Kafka](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Zookeeper_with_Kafka_1.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=8fc6c1523c32773254e40a7eb789f1a7)

### Zookeeper의 역할

| 역할 | 설명 |
|------|------|
| **Broker 추적** | 클러스터에 어떤 Broker가 있는지 관리 |
| **Leader 선출** | Partition Leader 결정 및 장애 시 교체 |
| **설정 저장** | Topic 설정, 접근 권한 등 |
| **변경 알림** | 새 Topic 생성, Broker 장애 등 이벤트 전달 |

### Zookeeper 클러스터 구성

- **홀수 개**의 서버로 구성 (3, 5, 7개)
- 과반수 투표(Quorum)를 통한 의사 결정
- 1개가 Leader(쓰기), 나머지는 Follower(읽기)

### 버전별 Zookeeper 필요 여부

| Kafka 버전 | Zookeeper |
|------------|-----------|
| 0.x ~ 2.x | **필수** |
| 3.x | 선택 (KRaft 모드 가능) |
| 4.x~ | **완전 제거** |

> ⚠️ **주의**: 현대 Kafka 개발에서는 클라이언트 애플리케이션에서
> **절대 Zookeeper에 직접 연결하지 마세요**. Kafka Broker만 사용하세요.

---
## ⚡ KRaft Mode

### KRaft란?

> **KRaft (Kafka Raft) = Zookeeper 없이 Kafka를 운영하는 새로운 방식**

2019년 KIP-500으로 제안되어, Kafka 3.3부터 **프로덕션 준비 완료**되었습니다.

![Kafka KRaft Mode vs Zookeeper](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_KRaft_Mode_1.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=d7227307e9586d4c0dd7c18313f18660)

### 왜 Zookeeper를 제거했나요?

| 문제 | 설명 |
|------|------|
| **파티션 제한** | Zookeeper로는 최대 약 200,000개 Partition만 지원 |
| **복잡한 설치** | Kafka + Zookeeper 두 시스템 운영 필요 |
| **메타데이터 동기화** | Kafka-Zookeeper 간 불일치 문제 발생 |
| **느린 복구** | Broker 변경 시 과도한 Leader 선출 발생 |
| **보안 모델** | Kafka보다 약한 보안 기능 |

### KRaft의 장점

| 장점 | 설명 |
|------|------|
| **수백만 Partition** | 확장성 대폭 향상 |
| **단순한 배포** | 단일 프로세스로 시작 |
| **빠른 복구** | Controller 장애 복구 속도 향상 |
| **통합 보안** | 일관된 보안 프레임워크 |

### KRaft 작동 방식

- Kafka Broker가 **Raft 프로토콜**을 직접 구현
- 메타데이터를 **내부 Kafka 로그**로 관리
- Controller들이 자체적으로 Leader를 선출

---
## 🔄 전체 데이터 흐름 정리

### 카카오톡 비유

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        Kafka = 카카오톡 단체 채팅방                        │
│                                                                         │
│    ┌──────────┐         ┌──────────┐         ┌──────────┐              │
│    │ Producer │ ──────▶ │  Topic   │ ──────▶ │ Consumer │              │
│    │ (발신자)  │  메시지  │ (채팅방)  │  메시지  │ (수신자)  │              │
│    └──────────┘  전송    └──────────┘  수신    └──────────┘              │
│                                                                         │
│    "hello"를           "hello"가           "hello"를                    │
│     입력하면            저장되고             읽어옴                       │
│                                                                         │
├─────────────────────────────────────────────────────────────────────────┤
│  💡 비유 정리:                                                           │
│     • Topic = 채팅방 (메시지가 모이는 곳)                                  │
│     • Partition = 채팅방 내 스레드 (병렬 처리용)                            │
│     • Producer = 메시지 보내는 사람                                       │
│     • Consumer = 메시지 읽는 사람                                         │
│     • Broker = 카카오 서버 (메시지를 저장하고 전달)                         │
│     • Consumer Group = 같은 앱을 쓰는 사람들 (작업 분담)                    │
└─────────────────────────────────────────────────────────────────────────┘
```

### 실제 시스템 예시

```
📦 쇼핑몰 주문 시스템:

┌──────────┐     ┌───────────────────┐     ┌──────────────┐
│ 주문 서버 │ ──▶ │ orders-topic      │ ──▶ │ 재고 시스템   │
│(Producer)│     │ ┌───────────────┐ │     │ (Consumer 1) │
└──────────┘     │ │ Partition 0   │ │     └──────────────┘
                 │ │ Partition 1   │ │
                 │ │ Partition 2   │ │     ┌──────────────┐
                 │ └───────────────┘ │ ──▶ │ 배송 시스템   │
                 │      (Broker)     │     │ (Consumer 2) │
                 └───────────────────┘     └──────────────┘

                                           ┌──────────────┐
                                       ──▶ │ 분석 시스템   │
                                           │ (Consumer 3) │
                                           └──────────────┘
```

---
## 🐳 Kafka Docker 이미지 종류

### 주요 이미지 비교

| 제공자 | 이미지 | 특징 | 추천 상황 |
|--------|--------|------|----------|
| **Apache** (공식) | `apache/kafka` | KRaft 모드 기본, 2023년~ | 학습, 공식 지원 원할 때 |
| **Confluent** | `confluentinc/cp-kafka` | 엔터프라이즈 기능 | 프로덕션, 고급 기능 필요 시 |
| **Bitnami** | `bitnami/kafka` | 보안 강화 + 쉬운 설정 | 빠른 설정, 보안 중시 |

### 우리가 사용할 이미지

```bash
# Apache 공식 이미지 (권장)
docker pull apache/kafka:latest
```

**선택 이유:**
- Apache 재단 공식 지원
- KRaft 모드 기본 (Zookeeper 불필요)
- 최신 기능 빠른 반영
- 학습 및 개발에 적합

---
## ❓ FAQ

**Q1. Kafka는 데이터베이스인가요?**

아닙니다. Kafka는 "이벤트 스트리밍 플랫폼"입니다.
- 데이터를 저장하지만, 쿼리(SELECT, JOIN 등) 기능은 없음
- 시간순으로 데이터를 저장하고 전달하는 것이 목적

---

**Q2. Producer가 보낸 메시지는 언제까지 보관되나요?**

설정에 따라 다릅니다 (기본값: 7일):
- `log.retention.hours`: 시간 기준 보관 기간
- `log.retention.bytes`: 용량 기준 보관 크기

---

**Q3. Consumer가 메시지를 읽으면 삭제되나요?**

**아닙니다!** 이것이 Kafka의 핵심 차별점입니다.
- 메시지를 읽어도 보관 기간까지 유지
- 여러 Consumer Group이 같은 메시지를 각자 읽을 수 있음
- offset을 되돌려서 재처리 가능

---

**Q4. Partition 수를 늘리면 항상 좋은가요?**

아닙니다. Trade-off가 있습니다:
- 장점: 병렬 처리 증가, 처리량 향상
- 단점: 메타데이터 증가, 리밸런싱 시간 증가, 파일 핸들 증가
- 권장: 예상 처리량에 맞게 적절히 설정

---

**Q5. Consumer Group 내 Consumer 수는 몇 개가 좋은가요?**

**Partition 수와 같거나 적게** 설정하세요.
- Consumer 수 > Partition 수 → 놀고 있는 Consumer 발생
- 이상적: Consumer 수 = Partition 수

---
## 📝 퀴즈

### Q1. Kafka에서 메시지를 생산하는 주체를 무엇이라 하나요?

- A) Broker
- B) Consumer
- C) Producer
- D) Topic

<details>
<summary>정답 보기</summary>

**정답: C) Producer**

Producer는 메시지를 생산해서 Kafka의 Topic으로 보내는 역할입니다.
</details>

---

### Q2. 같은 Key를 가진 메시지는 어떻게 되나요?

- A) 랜덤한 Partition으로 분배된다
- B) 항상 같은 Partition으로 전송된다
- C) 가장 비어있는 Partition으로 간다
- D) 첫 번째 Partition으로만 간다

<details>
<summary>정답 보기</summary>

**정답: B) 항상 같은 Partition으로 전송된다**

Key Hashing을 통해 같은 Key는 항상 같은 Partition으로 전송됩니다.
이를 통해 특정 Key에 대한 메시지 순서를 보장할 수 있습니다.
</details>

---

### Q3. Consumer Group에서 하나의 Partition은 몇 개의 Consumer가 읽을 수 있나요?

- A) 무제한
- B) 2개
- C) 1개
- D) Broker 수만큼

<details>
<summary>정답 보기</summary>

**정답: C) 1개**

Consumer Group 내에서 각 Partition은 **하나의 Consumer만** 읽을 수 있습니다.
단, 하나의 Consumer는 여러 Partition을 읽을 수 있습니다.
</details>

---

### Q4. Producer의 acks=all은 무엇을 의미하나요?

- A) 응답을 기다리지 않음
- B) Leader만 확인하면 됨
- C) 모든 ISR이 복제를 완료해야 성공
- D) 모든 Consumer가 읽어야 성공

<details>
<summary>정답 보기</summary>

**정답: C) 모든 ISR이 복제를 완료해야 성공**

acks=all은 가장 안전한 설정으로, Leader와 모든 In-Sync Replica가
메시지를 복제해야 성공으로 간주합니다.
</details>

---

### Q5. KRaft 모드의 가장 큰 장점은?

- A) 더 많은 보안 기능
- B) Zookeeper 없이 운영 가능
- C) 더 빠른 메시지 전송
- D) 더 적은 메모리 사용

<details>
<summary>정답 보기</summary>

**정답: B) Zookeeper 없이 운영 가능**

KRaft 모드는 Kafka가 자체적으로 메타데이터를 관리하여
별도의 Zookeeper 클러스터 없이 운영할 수 있게 해줍니다.
이로 인해 배포가 단순해지고, 수백만 개의 Partition까지 확장 가능합니다.
</details>

---
## 📋 핵심 요약

### Kafka 핵심 구성 요소

![The Ultimate Kafka 101 You Cannot Miss](https://substackcdn.com/image/fetch/$s_!QDAB!,w_1456,c_limit,f_webp,q_auto:good,fl_lossy/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F9d00da63-c92c-46c4-8bb7-86f0adf769c6_1354x1536.gif)

| 구성 요소 | 설명 |
|-----------|------|
| **Topic** | 메시지가 저장되는 주제별 카테고리 |
| **Partition** | Topic을 나눈 병렬 처리 단위 |
| **Offset** | Partition 내 메시지의 고유 위치 |
| **Producer** | 메시지를 보내는 애플리케이션 |
| **Consumer** | 메시지를 읽는 애플리케이션 |
| **Consumer Group** | 같은 역할의 Consumer 모음 |
| **Broker** | Kafka 서버 하나 |
| **Cluster** | Broker들의 집합 |

### 핵심 특징

| 특징 | 설명 |
|------|------|
| **Pull Model** | Consumer가 자신의 속도로 데이터를 가져감 |
| **Immutable** | 한번 쓰인 메시지는 수정/삭제 불가 |
| **Retention** | 설정 기간 동안 메시지 보관 (읽어도 삭제 안 됨) |
| **Replication** | 여러 Broker에 복제하여 장애 대비 |
| **Ordering** | Partition 내에서만 순서 보장 |

### Zookeeper vs KRaft

| 항목 | Zookeeper | KRaft |
|------|-----------|-------|
| 외부 의존성 | 필요 | 불필요 |
| 설치 복잡도 | 높음 | 낮음 |
| Partition 확장성 | ~200,000 | 수백만 |
| Kafka 4.x+ | 지원 안 함 | 기본 |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| Topic, Partition, Offset의 관계를 설명할 수 있나요? | ☐ |
| Producer와 Consumer의 역할을 구분할 수 있나요? | ☐ |
| Message Key가 Partition 분배에 미치는 영향을 아나요? | ☐ |
| Consumer Group과 Partition 할당 규칙을 이해했나요? | ☐ |
| Replication Factor와 ISR의 개념을 아나요? | ☐ |
| Producer acks 설정의 차이를 설명할 수 있나요? | ☐ |
| Zookeeper와 KRaft의 차이를 아나요? | ☐ |

---
## 📚 참고 자료

- [Conduktor Kafka Learn - Topics](https://docs.conduktor.io/learn/fundamentals/topics)
- [Conduktor Kafka Learn - Producers](https://docs.conduktor.io/learn/fundamentals/producers)
- [Conduktor Kafka Learn - Consumers](https://docs.conduktor.io/learn/fundamentals/consumers)
- [Conduktor Kafka Learn - Consumer Groups & Offsets](https://docs.conduktor.io/learn/fundamentals/consumer-groups-offsets)
- [Conduktor Kafka Learn - Brokers](https://docs.conduktor.io/learn/fundamentals/brokers)
- [Conduktor Kafka Learn - Topic Replication](https://docs.conduktor.io/learn/fundamentals/topic-replication)
- [Conduktor Kafka Learn - Zookeeper](https://docs.conduktor.io/learn/fundamentals/zookeeper)
- [Conduktor Kafka Learn - KRaft Mode](https://docs.conduktor.io/learn/fundamentals/kraft-mode)

---


# Day 06 - 4교시: Kafka QuickStart - 일단 돌려보기
> Docker로 Kafka를 실행하고, 메시지를 주고받는 전체 흐름 체험

## 🎯 학습 목표
이 파트를 마치면 다음을 할 수 있습니다:

- Docker로 Kafka 브로커를 실행할 수 있다
- `docker exec`로 컨테이너 내부에 접속할 수 있다
- Kafka CLI로 토픽(Topic)을 생성하고 조회할 수 있다
- Producer로 메시지를 보내고, Consumer로 메시지를 읽을 수 있다
- `--from-beginning` 옵션의 역할을 이해한다

---

## 📚 전체 실습 흐름

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        우리가 만들 것                                    │
│                                                                         │
│    ┌──────────┐         ┌──────────┐         ┌──────────┐              │
│    │ Producer │ ──────▶ │  Topic   │ ──────▶ │ Consumer │              │
│    │ (발신자)   │  메시지  │ (우편함)   │  메시지   │ (수신자)  │              │
│    └──────────┘  전송    └──────────┘  수신    └──────────┘              │
│                                                                         │
│    "hello"를           "hello"가           "hello"를                    │
│     입력하면            저장되고             읽어옴                       │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 실습 순서

```
Step 1: Kafka 브로커 실행 (docker run)
    ↓
Step 2: 컨테이너 내부 접속 (docker exec)
    ↓
Step 3: 토픽 생성 (kafka-topics.sh)
    ↓
Step 4: 메시지 보내기 (kafka-console-producer.sh)
    ↓
Step 5: 메시지 읽기 (kafka-console-consumer.sh)
    ↓
Step 6: 정리 (docker rm)
```

---
## 🚀 Step 1: Kafka 브로커 실행하기

### 왜 이걸 하나요?

Kafka를 사용하려면 먼저 **Kafka 서버(브로커)**가 실행되어 있어야 합니다.
마치 카카오톡을 쓰려면 카카오 서버가 돌아가고 있어야 하는 것처럼!

### 실행 환경

```
┌─────────────────────────────────────────────────────────────────┐
│  🖥️ 호스트 (내 컴퓨터) 터미널                                     │
│  프롬프트 예시: user@mypc:~$                                     │
└─────────────────────────────────────────────────────────────────┘
```

### 명령어

```bash
# ============================================================
# Kafka 브로커 컨테이너 실행
# ============================================================
# 호스트 터미널에서 실행

docker run -d --name broker apache/kafka:latest
```

### 명령어 상세 해석

```bash
docker run                    # 새 컨테이너 생성 및 실행
  -d                          # Detached 모드: 백그라운드에서 실행
                              # → 터미널이 막히지 않고 계속 사용 가능
  --name broker               # 컨테이너에 "broker"라는 이름 부여
                              # → 나중에 이 이름으로 접근/관리
  apache/kafka:latest         # 사용할 이미지
                              # → Apache 공식 Kafka 최신 버전
                              # → 처음 실행 시 Docker Hub에서 자동 다운로드
```

| 옵션 | 의미 | 비유 |
|------|------|------|
| `docker run` | 새 컨테이너 생성 및 실행 | "새 앱 설치하고 실행" |
| `-d` | 백그라운드 실행 | "창 숨기고 뒤에서 돌리기" |
| `--name broker` | 컨테이너 이름 지정 | "이 앱을 'broker'라고 부르자" |
| `apache/kafka:latest` | 사용할 이미지 | "Apache 공식 Kafka 최신 버전" |

### 실행 확인

```bash
# 컨테이너가 잘 실행 중인지 확인
docker ps
```

**실행 결과:**
```
CONTAINER ID   IMAGE                 STATUS          NAMES
b9e839e7b7ab   apache/kafka:latest   Up 21 seconds   broker
```

### 지금 상태 시각화

```
┌─────────────────────────────────────────────────────────────────┐
│  🖥️ 내 컴퓨터 (호스트)                                              │
│                                                                 │
│    ┌─────────────────────────────────────────────────────┐     │
│    │  📦 Docker 컨테이너: broker                          │     │
│    │  ┌─────────────────────────────────────────────┐   │     │
│    │  │                                             │   │     │
│    │  │   🟢 Kafka 브로커 실행 중!                     │   │     │
│    │  │      (포트 9092에서 대기)                      │   │     │
│    │  │                                             │   │     │
│    │  │   📁 /opt/kafka/bin/ ← Kafka CLI 도구 위치    │   │     │
│    │  │                                             │   │     │
│    │  └─────────────────────────────────────────────┘   │     │
│    └─────────────────────────────────────────────────────┘     │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---
## 🚀 Step 2: 컨테이너 내부로 들어가기

### 왜 이걸 하나요?

Kafka CLI 도구(`kafka-topics.sh` 등)가 **컨테이너 안에** 있기 때문입니다.
호스트 터미널에서는 이 명령어들을 찾을 수 없어요!

```
┌─────────────────────────────────────────────────────────────────┐
│  ❌ 호스트에서 직접 실행하면?                                      │
│                                                                 │
│  user@mypc:~$ kafka-topics.sh                                   │
│  → "command not found" (명령어를 찾을 수 없음)                    │
│                                                                 │
│  ✅ 그래서 컨테이너 안으로 들어가서 실행해야 함!                     │
└─────────────────────────────────────────────────────────────────┘
```

### 터미널 환경 변화

```
실행 전:                              실행 후:
┌─────────────────────┐              ┌─────────────────────┐
│ 🖥️ 호스트 터미널     │              │ 📦 컨테이너 내부 쉘   │
│                     │   docker     │                     │
│ user@mypc:~$        │ ──────────▶  │ /opt/kafka/bin $    │
│                     │   exec       │                     │
│ (내 컴퓨터)         │              │ (Kafka 컨테이너 안)  │
└─────────────────────┘              └─────────────────────┘
```

### 명령어

```bash
# ============================================================
# 실행 중인 컨테이너 내부로 접속
# ============================================================
# 호스트 터미널에서 실행

docker exec --workdir /opt/kafka/bin/ -it broker sh
```

### 명령어 상세 해석

```bash
docker exec                   # 실행 중인 컨테이너에서 명령 실행
  --workdir /opt/kafka/bin/   # 시작 디렉토리를 Kafka 도구 폴더로 지정
                              # → 이 폴더에 kafka-topics.sh 등이 있음
  -i                          # Interactive: 표준 입력 유지 (키보드 입력 가능)
  -t                          # TTY: 터미널 할당 (프롬프트 표시)
                              # → -it로 합쳐서 사용 (대화형 모드)
  broker                      # 접속할 컨테이너 이름
  sh                          # 실행할 쉘 프로그램
                              # → bash 대신 sh 사용 (경량 이미지)
```

| 옵션 | 의미 |
|------|------|
| `docker exec` | 실행 중인 컨테이너에서 명령 실행 |
| `--workdir /opt/kafka/bin/` | 시작 위치를 Kafka 도구 폴더로 지정 |
| `-it` | 대화형 터미널 모드 (키보드 입력 가능) |
| `broker` | 접속할 컨테이너 이름 |
| `sh` | 실행할 쉘 프로그램 |

### 성공 확인

프롬프트가 바뀌면 성공입니다:
```bash
# 이렇게 바뀌면 컨테이너 안에 들어온 것!
/opt/kafka/bin $
```

> 💡 **Tip**: 컨테이너에서 나오려면 `exit` 입력 또는 `Ctrl+D`

---
## 🚀 Step 3: 토픽(Topic) 생성하기

### 왜 이걸 하나요?

메시지를 보내려면 **어디로** 보낼지 정해야 합니다.
토픽은 메시지가 저장되는 **"주제별 우편함"**입니다.

```
┌─────────────────────────────────────────────────────────────────┐
│  💡 토픽 = 메시지 분류함                                          │
│                                                                 │
│  예) 쇼핑몰 서비스라면:                                           │
│    • "orders" 토픽 → 주문 관련 메시지                             │
│    • "payments" 토픽 → 결제 관련 메시지                           │
│    • "notifications" 토픽 → 알림 관련 메시지                      │
│                                                                 │
│  지금은 연습용으로 "test-topic"을 만들어볼게요!                     │
└─────────────────────────────────────────────────────────────────┘
```

### 실행 환경

```
┌─────────────────────────────────────────────────────────────────┐
│  📦 컨테이너 내부 쉘 (앞에서 docker exec로 들어온 상태)             │
│  프롬프트: /opt/kafka/bin $                                      │
└─────────────────────────────────────────────────────────────────┘
```

### 명령어

```bash
# ============================================================
# 새 토픽 생성
# ============================================================
# 컨테이너 내부에서 실행

./kafka-topics.sh --bootstrap-server localhost:9092 --create --topic test-topic
```

### 명령어 상세 해석

```bash
./kafka-topics.sh              # 토픽 관리 CLI 도구 실행
                               # ./ = 현재 디렉토리에서 실행

  --bootstrap-server           # Kafka 브로커 주소 지정
    localhost:9092             # localhost: 같은 컨테이너 내부
                               # 9092: Kafka 기본 포트

  --create                     # 토픽 생성 명령
                               # 다른 옵션: --list, --describe, --delete

  --topic test-topic           # 생성할 토픽 이름
                               # 규칙: 영문, 숫자, 하이픈(-), 언더스코어(_)
```

| 옵션 | 의미 |
|------|------|
| `./kafka-topics.sh` | 토픽 관리 도구 실행 |
| `--bootstrap-server localhost:9092` | Kafka 서버 주소 (기본 포트 9092) |
| `--create` | "새로 만들어줘" |
| `--topic test-topic` | 토픽 이름은 "test-topic" |

### 생성 확인

```bash
# 생성된 토픽 목록 확인
./kafka-topics.sh --bootstrap-server localhost:9092 --list
```

**실행 결과:**
```
Created topic test-topic.
```

```bash
# 생성된 토픽 목록 확인
./kafka-topics.sh --bootstrap-server localhost:9092 --list
```

**실행 결과:**
```
test-topic
```

### 토픽 상세 정보 확인 (선택)

```bash
# 토픽의 상세 정보 조회
./kafka-topics.sh --bootstrap-server localhost:9092 --describe --topic test-topic
```

**실행 결과:**
```
Topic: test-topic  TopicId: G0nR1SsnQ_ixQ0ErDur_tA  PartitionCount: 1  ReplicationFactor: 1  Configs: segment.bytes=1073741824
    Topic: test-topic  Partition: 0  Leader: 1  Replicas: 1  Isr: 1
```

> **출력 해석**:
> - `PartitionCount: 1` - 파티션 1개 (기본값)
> - `ReplicationFactor: 1` - 복제본 1개 (단일 브로커)
> - `Leader: 1` - 브로커 1번이 리더

### 현재 상태 시각화

```
┌─────────────────────────────────────────────────────────────────┐
│  📦 Kafka 브로커 (컨테이너 내부)                                  │
│                                                                 │
│    ┌─────────────────────────────────────────────────────┐     │
│    │                    토픽 목록                         │     │
│    │  ┌───────────────────────────────────────────────┐  │     │
│    │  │  📮 test-topic                                │  │     │
│    │  │     ├── 파티션: 1개 (기본값)                   │  │     │
│    │  │     └── 메시지: 0개 (아직 비어있음)            │  │     │
│    │  └───────────────────────────────────────────────┘  │     │
│    └─────────────────────────────────────────────────────┘     │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---
## 🚀 Step 4: 메시지 보내기 (Producer)

### 왜 이걸 하나요?

토픽(우편함)을 만들었으니, 이제 **메시지를 보내볼** 차례입니다!
Producer는 메시지를 **생산**해서 토픽에 **전송**하는 역할입니다.

### 실행 환경

```
┌─────────────────────────────────────────────────────────────────┐
│  📦 컨테이너 내부 쉘                                              │
│  프롬프트: /opt/kafka/bin $                                      │
└─────────────────────────────────────────────────────────────────┘
```

### 명령어

```bash
# ============================================================
# Producer 실행 (메시지 전송 모드)
# ============================================================
# 컨테이너 내부에서 실행

./kafka-console-producer.sh --bootstrap-server localhost:9092 --topic test-topic
```

### 명령어 상세 해석

```bash
./kafka-console-producer.sh    # 콘솔 기반 Producer 실행
                               # 대화형으로 메시지 입력 가능

  --bootstrap-server           # Kafka 브로커 주소
    localhost:9092

  --topic test-topic           # 메시지를 보낼 대상 토픽
                               # 이 토픽에 메시지가 저장됨
```

### ⚠️ 중요: 인터랙티브 모드 이해하기

명령어를 실행하면 **대화형 입력 모드**로 전환됩니다:

```
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│  명령어 실행 전:                                                  │
│  /opt/kafka/bin $ ./kafka-console-producer.sh ...               │
│                                                                 │
│  명령어 실행 후:                                                  │
│  >                    ← 이 프롬프트가 나타남!                     │
│                         여기에 입력하는 모든 것이 메시지가 됨       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 메시지 보내기 실습

`>` 프롬프트가 나타나면 다음을 입력하세요:

```
> hello                    ← 입력 후 Enter
> world                    ← 입력 후 Enter
> Kafka is awesome!        ← 입력 후 Enter
```

### 메시지 흐름 시각화

```
┌─────────────────────────────────────────────────────────────────┐
│  Producer 인터랙티브 모드                                         │
│                                                                 │
│  > hello                    ← 입력 후 Enter                     │
│  > world                    ← 입력 후 Enter                     │
│  > Kafka is awesome!        ← 입력 후 Enter                     │
│  >                                                              │
│  (종료하려면 Ctrl+C)                                              │
│                                                                 │
├─────────────────────────────────────────────────────────────────┤
│  💡 지금 일어나는 일:                                             │
│                                                                 │
│    입력: "hello" + Enter                                        │
│           │                                                     │
│           ▼                                                     │
│    ┌─────────────────┐         ┌─────────────────┐             │
│    │   Producer      │ ──────▶ │   test-topic    │             │
│    │   (내가 입력)    │  전송   │   [hello]       │             │
│    └─────────────────┘         └─────────────────┘             │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 시간순 메시지 저장 과정

```
시간순으로 보면:

  t=0   입력: "hello" + Enter
        ─────────────────────────────────▶
                                         │
                                         ▼
                              ┌──────────────────┐
                              │   test-topic     │
                              │  ┌────────────┐  │
                              │  │ 0: hello   │  │
                              │  └────────────┘  │
                              └──────────────────┘

  t=1   입력: "world" + Enter
        ─────────────────────────────────▶
                                         │
                                         ▼
                              ┌──────────────────┐
                              │   test-topic     │
                              │  ┌────────────┐  │
                              │  │ 0: hello   │  │
                              │  │ 1: world   │  │
                              │  └────────────┘  │
                              └──────────────────┘

  t=2   입력: "Kafka is awesome!" + Enter
        ─────────────────────────────────▶
                                         │
                                         ▼
                              ┌─────────────────────────┐
                              │   test-topic            │
                              │  ┌───────────────────┐  │
                              │  │ 0: hello          │  │
                              │  │ 1: world          │  │
                              │  │ 2: Kafka is ...   │  │
                              │  └───────────────────┘  │
                              └─────────────────────────┘
```

**메시지 보내기 완료 후 `Ctrl+C`로 Producer를 종료하세요.**

---
## 🚀 Step 5: 메시지 읽기 (Consumer)

### 왜 이걸 하나요?

토픽에 메시지를 보냈으니, 이제 **메시지를 읽어볼** 차례입니다!
Consumer는 토픽에서 메시지를 **소비(읽기)**하는 역할입니다.

### 명령어

```bash
# ============================================================
# Consumer 실행 (메시지 수신 모드)
# ============================================================
# 컨테이너 내부에서 실행

./kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic test-topic --from-beginning
```

### 명령어 상세 해석

```bash
./kafka-console-consumer.sh    # 콘솔 기반 Consumer 실행
                               # 토픽의 메시지를 화면에 출력

  --bootstrap-server           # Kafka 브로커 주소
    localhost:9092

  --topic test-topic           # 메시지를 읽을 대상 토픽

  --from-beginning             # ⚠️ 핵심 옵션!
                               # 토픽의 처음부터 모든 메시지 읽기
                               # 이 옵션 없으면 "지금부터" 오는 것만 읽음
```

### 핵심 옵션: `--from-beginning`

```
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│  --from-beginning 없이 실행하면:                                  │
│  → "지금부터" 들어오는 새 메시지만 표시                             │
│  → 이미 토픽에 있던 메시지는 안 보임                               │
│                                                                 │
│  --from-beginning 포함하면:                                       │
│  → 토픽의 "처음부터" 모든 메시지 표시                              │
│  → 과거 메시지도 모두 볼 수 있음                                   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

| 옵션 | 동작 |
|------|------|
| `--from-beginning` 있음 | 처음부터 모든 메시지 읽기 (과거 + 현재) |
| `--from-beginning` 없음 | 지금부터 새 메시지만 읽기 (실시간) |

### 실행 결과

```
hello
world
Kafka is awesome!
Processed a total of 3 messages
```

이전에 Producer로 보낸 메시지들이 그대로 출력됩니다!
마지막에 "Processed a total of 3 messages"는 몇 개의 메시지를 처리했는지 보여줍니다.

**메시지 확인 후 `Ctrl+C`로 Consumer를 종료하세요.**

### 전체 흐름 완성!

```
┌─────────────────────────────────────────────────────────────────────────┐
│                       완성된 메시지 흐름                                  │
│                                                                         │
│   ┌──────────────┐                                ┌──────────────┐      │
│   │   Producer   │                                │   Consumer   │      │
│   │              │                                │              │      │
│   │  > hello     │──┐                         ┌──▶│  hello       │      │
│   │  > world     │  │     ┌──────────────┐    │   │  world       │      │
│   │              │  └────▶│  test-topic  │────┘   │  Kafka is... │      │
│   │              │        │  ┌─────────┐ │        │              │      │
│   │              │        │  │ hello   │ │        │              │      │
│   │              │        │  │ world   │ │        │              │      │
│   │              │        │  │ Kafka.. │ │        │              │      │
│   │              │        │  └─────────┘ │        │              │      │
│   └──────────────┘        └──────────────┘        └──────────────┘      │
│                                                                         │
│        발신                    저장                    수신              │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## 🚀 Step 6: 정리하기

### 왜 이걸 하나요?

실습이 끝났으니 사용한 리소스를 정리합니다.
컨테이너를 삭제하지 않으면 계속 리소스(CPU, 메모리)를 점유합니다!

### ⚠️ 주의: 호스트에서 실행!

```
┌─────────────────────────────────────────────────────────────────┐
│  ⚠️ 컨테이너 안에서 exit로 먼저 나온 후, 호스트에서 실행!          │
│                                                                 │
│  /opt/kafka/bin $ exit          ← 컨테이너에서 나오기            │
│  user@mypc:~$                   ← 호스트로 돌아옴                │
└─────────────────────────────────────────────────────────────────┘
```

### 명령어

```bash
# ============================================================
# 컨테이너 강제 삭제
# ============================================================
# 호스트 터미널에서 실행

docker rm -f broker
```

### 명령어 상세 해석

```bash
docker rm                      # 컨테이너 삭제 명령
  -f                           # Force: 실행 중이어도 강제 삭제
                               # → 이 옵션 없으면 stop 후 rm 필요
  broker                       # 삭제할 컨테이너 이름
```

### 삭제 확인

```bash
# 컨테이너 목록 확인 (broker가 없어야 함)
docker ps -a
```

---
## 🧪 실습: 실시간 메시지 주고받기

### 목표

터미널 2개를 열어서 **Producer와 Consumer를 동시에 실행**하고,
메시지가 **실시간으로** 전달되는 것을 확인합니다.

### 사전 준비

```bash
# 호스트에서 Kafka 브로커 다시 실행
docker run -d --name broker apache/kafka:latest

# 토픽 생성 (컨테이너 외부에서 한 줄로 실행)
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 --create --topic chat-room
```

### 터미널 1: Producer 실행

```bash
# ============================================================
# 터미널 1: Producer (메시지 보내기)
# ============================================================

docker exec -it broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 --topic chat-room
```

### 터미널 2: Consumer 실행 (새 터미널 열기)

```bash
# ============================================================
# 터미널 2: Consumer (실시간 메시지 받기)
# ============================================================
# ⚠️ --from-beginning 없이 실행 → 실시간 메시지만 수신!

docker exec -it broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 --topic chat-room
```

### 테스트

```
┌─────────────────────────────────────────────────────────────────┐
│  터미널 1 (Producer)          터미널 2 (Consumer)               │
│                                                                 │
│  > 안녕하세요                  (대기 중...)                      │
│                               안녕하세요         ← 실시간 출력!  │
│                                                                 │
│  > Kafka 재밌네요!             (대기 중...)                      │
│                               Kafka 재밌네요!   ← 실시간 출력!  │
│                                                                 │
│  > ^C (종료)                   ^C (종료)                        │
└─────────────────────────────────────────────────────────────────┘
```

> 💡 **핵심 포인트**: `--from-beginning` 없이 실행하면 "지금부터" 오는 메시지만 받습니다!

---
## 📝 Kafka CLI 명령어 정리

### 토픽 관련 (kafka-topics.sh)

```bash
# 토픽 생성
./kafka-topics.sh --bootstrap-server localhost:9092 --create --topic [토픽명]

# 토픽 목록 조회
./kafka-topics.sh --bootstrap-server localhost:9092 --list

# 토픽 상세 정보
./kafka-topics.sh --bootstrap-server localhost:9092 --describe --topic [토픽명]

# 토픽 삭제
./kafka-topics.sh --bootstrap-server localhost:9092 --delete --topic [토픽명]
```

### Producer 관련 (kafka-console-producer.sh)

```bash
# 기본 실행 (대화형 입력)
./kafka-console-producer.sh --bootstrap-server localhost:9092 --topic [토픽명]

# 파일에서 읽어서 전송
./kafka-console-producer.sh --bootstrap-server localhost:9092 --topic [토픽명] < file.txt
```

### Consumer 관련 (kafka-console-consumer.sh)

```bash
# 실시간 메시지만 수신
./kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic [토픽명]

# 처음부터 모든 메시지 수신
./kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic [토픽명] --from-beginning

# Consumer 그룹 지정 (나중에 배움)
./kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic [토픽명] --group [그룹명]
```

---
## ❓ FAQ

**Q1. `docker exec` vs `docker run`의 차이는?**

- `docker run`: 새 컨테이너를 **생성하고 실행**
- `docker exec`: **이미 실행 중인** 컨테이너에서 명령 실행

```bash
# 새 컨테이너 생성
docker run -d --name broker apache/kafka:latest

# 실행 중인 컨테이너에 접속
docker exec -it broker sh
```

---

**Q2. localhost:9092에서 9092는 뭔가요?**

9092는 Kafka의 기본 포트 번호입니다.
- 웹 서버: 80 (HTTP), 443 (HTTPS)
- MySQL: 3306
- PostgreSQL: 5432
- Kafka: **9092**

---

**Q3. 토픽을 삭제하면 메시지도 사라지나요?**

네, 토픽을 삭제하면 그 안의 모든 메시지도 함께 삭제됩니다.

```bash
# 토픽 삭제 (메시지도 함께 삭제!)
./kafka-topics.sh --bootstrap-server localhost:9092 --delete --topic test-topic
```

---

**Q4. Producer가 보낸 메시지는 언제까지 남아있나요?**

Kafka 기본 설정으로 **7일간** 보관됩니다.
- `log.retention.hours=168` (168시간 = 7일)
- 설정 변경으로 보관 기간 조정 가능

---

**Q5. 왜 컨테이너 안에 들어가서 명령어를 실행해야 하나요?**

Kafka CLI 도구들이 컨테이너 내부에 설치되어 있기 때문입니다.
호스트에 직접 Kafka를 설치하면 호스트에서도 실행 가능하지만,
Docker 환경에서는 컨테이너 내부에 도구가 있습니다.

> 💡 다음 시간에 배울 Kafka UI를 사용하면 CLI 없이도 관리할 수 있어요!

---
## 📝 퀴즈

### Q1. Kafka 컨테이너 내부로 들어가는 명령어는?

- A) docker run -it broker sh
- B) docker exec -it broker sh
- C) docker start -it broker sh
- D) docker attach broker sh

<details>
<summary>정답 보기</summary>

**정답: B) docker exec -it broker sh**

`docker exec`는 이미 실행 중인 컨테이너에서 명령을 실행합니다.
`docker run`은 새 컨테이너를 생성합니다.
</details>

---

### Q2. 토픽을 생성하는 kafka-topics.sh 옵션은?

- A) --new
- B) --make
- C) --create
- D) --add

<details>
<summary>정답 보기</summary>

**정답: C) --create**

`./kafka-topics.sh --bootstrap-server localhost:9092 --create --topic 토픽명`
으로 토픽을 생성합니다.
</details>

---

### Q3. Consumer에서 과거 메시지까지 모두 읽으려면 어떤 옵션이 필요한가요?

- A) --all-messages
- B) --from-beginning
- C) --include-history
- D) --read-all

<details>
<summary>정답 보기</summary>

**정답: B) --from-beginning**

이 옵션 없이 Consumer를 실행하면 "지금부터" 오는 새 메시지만 읽습니다.
`--from-beginning`을 붙이면 토픽의 처음부터 모든 메시지를 읽습니다.
</details>

---

### Q4. Kafka의 기본 포트 번호는?

- A) 8080
- B) 3000
- C) 9092
- D) 5432

<details>
<summary>정답 보기</summary>

**정답: C) 9092**

Kafka 브로커는 기본적으로 9092 포트에서 클라이언트 연결을 기다립니다.
`--bootstrap-server localhost:9092`로 연결합니다.
</details>

---

### Q5. 다음 중 Producer의 역할은?

- A) 토픽을 생성한다
- B) 메시지를 토픽에 전송한다
- C) 토픽에서 메시지를 읽는다
- D) Kafka 서버를 관리한다

<details>
<summary>정답 보기</summary>

**정답: B) 메시지를 토픽에 전송한다**

Producer는 메시지를 생산(produce)하여 토픽에 보내는 역할입니다.
메시지를 읽는 것은 Consumer의 역할입니다.
</details>

---
## 📋 과제

### 과제 1: 새로운 토픽 만들기 (난이도: ⭐)

1. Kafka 브로커를 실행하세요
2. `my-first-topic`이라는 이름의 토픽을 생성하세요
3. 토픽 목록을 조회하여 생성을 확인하세요

<details>
<summary>💡 힌트</summary>

- 브로커 실행: `docker run -d --name broker apache/kafka:latest`
- 토픽 생성 명령어: `kafka-topics.sh --create --topic [토픽명]`
- 토픽 확인 명령어: `kafka-topics.sh --list`
- 컨테이너 내부에서 실행하거나 `docker exec`로 직접 실행할 수 있습니다
</details>

<details>
<summary>✅ 모범답안</summary>

```bash
# 1. 브로커 실행
docker run -d --name broker apache/kafka:latest

# 2. 토픽 생성 (컨테이너 외부에서 한 줄로 실행)
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 --create --topic my-first-topic

# 3. 토픽 확인
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 --list
```

**실행 결과:**
```
Created topic my-first-topic.
```

```
my-first-topic
test-topic
```
</details>

---

### 과제 2: 대화 시뮬레이션 (난이도: ⭐⭐)

1. `chat-room`이라는 토픽을 생성하세요
2. 터미널 창을 **2개** 열어서:
   - 터미널 1: Producer 실행 (메시지 보내기)
   - 터미널 2: Consumer 실행 (메시지 받기)
3. Producer에서 메시지를 보내면 Consumer에서 **실시간으로** 표시되는지 확인하세요

<details>
<summary>💡 힌트</summary>

- 토픽 생성: `kafka-topics.sh --create --topic chat-room`
- Producer: `kafka-console-producer.sh --topic chat-room`
- Consumer: `kafka-console-consumer.sh --topic chat-room`
- **핵심**: Consumer를 `--from-beginning` **없이** 실행하면 실시간 메시지만 받습니다!
</details>

<details>
<summary>✅ 모범답안</summary>

```bash
# 0. 토픽 생성
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 --create --topic chat-room

# 터미널 1: Producer
docker exec -it broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 --topic chat-room

# 터미널 2: Consumer (--from-beginning 없이!)
docker exec -it broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 --topic chat-room
```

**테스트 결과:**
```
터미널 1 (Producer)          터미널 2 (Consumer)
> 안녕하세요                  (대기 중...)
                             안녕하세요         ← 실시간 출력!
> Kafka 재밌네요!             (대기 중...)
                             Kafka 재밌네요!   ← 실시간 출력!
```
</details>

---

### 과제 3: 메시지 재처리 체험 (난이도: ⭐⭐)

Kafka의 핵심 특징 중 하나는 **메시지 재처리**가 가능하다는 것입니다.

1. 토픽에 메시지 5개를 보내세요
2. Consumer로 메시지를 모두 읽으세요 (`--from-beginning`)
3. Consumer를 종료했다가 다시 실행하세요 (`--from-beginning`)
4. 질문: 메시지가 다시 보이나요? → **YES!** (재처리 가능)

<details>
<summary>💡 힌트</summary>

- 새 토픽 생성: `kafka-topics.sh --create --topic reprocess-test`
- Producer로 메시지 5개 입력 후 `Ctrl+C`로 종료
- Consumer를 `--from-beginning`으로 실행
- Consumer를 종료 후 다시 `--from-beginning`으로 실행
- **핵심**: Consumer가 읽어도 메시지는 삭제되지 않습니다!
</details>

<details>
<summary>✅ 모범답안</summary>

```bash
# 1. 테스트용 토픽 생성
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 --create --topic reprocess-test

# 2. Producer로 메시지 5개 전송
docker exec -it broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 --topic reprocess-test
# > 메시지 1
# > 메시지 2
# > 메시지 3
# > 메시지 4
# > 메시지 5
# Ctrl+C로 종료

# 3. 첫 번째 Consumer 실행
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 --topic reprocess-test --from-beginning
```

**첫 번째 실행 결과:**
```
메시지 1
메시지 2
메시지 3
메시지 4
메시지 5
Processed a total of 5 messages
```

```bash
# 4. Consumer를 종료 후 다시 실행
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 --topic reprocess-test --from-beginning
```

**두 번째 실행 결과:**
```
메시지 1
메시지 2
메시지 3
메시지 4
메시지 5
Processed a total of 5 messages
```

> **결론**: 메시지가 **다시 모두 출력**됩니다! 이것이 Kafka의 재처리 기능입니다.
> Consumer가 읽어도 메시지는 보관 기간(기본 7일)까지 유지됩니다.
</details>

---

### 보너스 과제: 토픽 상세 정보 확인 (난이도: ⭐⭐⭐)

`--describe` 옵션으로 토픽의 상세 정보를 확인하고, 다음 질문에 답하세요:

- **PartitionCount**는 몇 개인가요?
- **ReplicationFactor**는 몇인가요?
- **Leader**는 어떤 브로커인가요?

<details>
<summary>💡 힌트</summary>

- 명령어: `kafka-topics.sh --describe --topic [토픽명]`
- 출력에서 `PartitionCount`, `ReplicationFactor`, `Leader` 값을 찾으세요
</details>

<details>
<summary>✅ 모범답안</summary>

```bash
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 --describe --topic test-topic
```

**실행 결과:**
```
Topic: test-topic  TopicId: G0nR1SsnQ_ixQ0ErDur_tA  PartitionCount: 1  ReplicationFactor: 1  Configs: segment.bytes=1073741824
    Topic: test-topic  Partition: 0  Leader: 1  Replicas: 1  Isr: 1
```

**정답:**
- **PartitionCount**: 1개 (기본값)
- **ReplicationFactor**: 1 (단일 브로커이므로 복제 없음)
- **Leader**: 브로커 1번

> 💡 이 개념들은 Day 07에서 자세히 배웁니다!
</details>

---
## 📋 핵심 요약

### 실습 흐름 정리

| 단계 | 명령어 | 설명 |
|------|--------|------|
| 1. 브로커 실행 | `docker run -d --name broker apache/kafka:latest` | Kafka 서버 시작 |
| 2. 컨테이너 접속 | `docker exec -it broker sh` | CLI 도구 사용 위해 접속 |
| 3. 토픽 생성 | `kafka-topics.sh --create --topic test-topic` | 메시지 저장소 생성 |
| 4. 메시지 보내기 | `kafka-console-producer.sh --topic test-topic` | Producer로 전송 |
| 5. 메시지 읽기 | `kafka-console-consumer.sh --topic test-topic` | Consumer로 수신 |
| 6. 정리 | `docker rm -f broker` | 컨테이너 삭제 |

### 핵심 명령어

```bash
# 토픽 관리
kafka-topics.sh --create/--list/--describe/--delete

# Producer
kafka-console-producer.sh --topic [토픽명]

# Consumer
kafka-console-consumer.sh --topic [토픽명] [--from-beginning]
```

### --from-beginning 옵션

| 옵션 | 동작 |
|------|------|
| 있음 | 처음부터 모든 메시지 (과거 + 현재) |
| 없음 | 지금부터 새 메시지만 (실시간) |

---

## ✅ 학습 후 체크리스트

| 질문 | 확인 |
|------|------|
| Docker로 Kafka 브로커를 실행할 수 있나요? | ☐ |
| `docker exec`로 컨테이너 내부에 접속할 수 있나요? | ☐ |
| 토픽을 생성하고 목록을 조회할 수 있나요? | ☐ |
| Producer로 메시지를 보낼 수 있나요? | ☐ |
| Consumer로 메시지를 읽을 수 있나요? | ☐ |
| `--from-beginning` 옵션의 역할을 설명할 수 있나요? | ☐ |

---

## 🚀 다음 시간 예고

**Day 07: Kafka 실습과 클러스터**

- Kafka UI로 시각적으로 관리하기
- Docker Compose로 여러 서비스 한번에 실행
- 설정 커스터마이징
- Python에서 Kafka 연결하기